Download das imagens do ArborMap, selecionando as que possuem menos de 20% de nuvens

In [ ]:
import ee
import geemap
import numpy as np
import os
import json
import time
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
IS_FREE_COLAB = True
FILES_DIR = "."

if IS_FREE_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    FILES_DIR = "/content/drive/MyDrive/IA/Pibic/Vulnerability"

ee.Authenticate()
ee.Initialize(project="script-39a43")

# ============================================================
# CIDADES
#
# Critérios de seleção:
#   - Variabilidade de porte (5k–900k hab)
#   - Variabilidade de altitude (20 m–1 000 m)
#   - Diferentes biomas do Nordeste (Caatinga, Cerrado, Litoral)
#   - Cidades do Piauí originais mantidas
#   - Adicionadas cidades de outros estados nordestinos para
#     melhorar generalização sem sair do clima semiárido/tropical
# ============================================================
CIDADES = {
    # ── Piauí (originais) ──────────────────────────────────────
    # "Teresina":               (-42.790, -5.080),   # ~900k hab, vale do Parnaíba
    # "Parnaiba":               (-41.770, -2.930),   # litoral, brisa marítima
    # "Picos":                  (-41.460, -7.080),   # sertão, altitude ~200 m
    # "Floriano":               (-43.010, -6.780),   # sertão sul
    # "Bom_Jesus":              (-44.375, -9.075),   # cerrado piauiense
    # "Sao_Raimundo_Nonato":    (-42.705, -8.975),   # caatinga, ~30k hab
    # "Corrente":               (-45.125,-10.435),   # extremo sul, cerrado
    # "Urucui":                 (-44.450, -7.200),   # agronegócio, chapada
    # "Pedro_II":               (-41.425, -4.415),   # altitude ~970 m — outlier térmico valioso

    # # ── Nordeste — diversidade climática e urbana ───────────────
    # "Vitoria_da_Conquista_BA":(-40.844, -14.866),  # ~340k hab, altitude 923 m — mais frio
    # "Juazeiro_do_Norte_CE":   (-39.315, -7.213),   # ~270k hab, caatinga seca
    # "Mossoro_RN":             (-37.344, -5.188),   # ~300k hab, salineiras, muito árido
    # "Imperatriz_MA":          (-47.492, -5.527),   # ~260k hab, transição cerrado-amazônia
    # "Barreiras_BA":           (-44.993, -12.144),  # ~140k hab, MATOPIBA, cerrado intenso
    # "Paulo_Afonso_BA":        (-38.215, -9.400),   # ~110k hab, caatinga + Rio São Francisco
    # "Sobral_CE":              (-40.348, -3.686),   # ~210k hab, sertão cearense

    # Cidades adicionadas
    "Araguaina_TO":             (-48.206, -7.191),   # ~180k hab, transição cerrado-amazônia (Tocantins)
    "Patos_PB":                 (-37.281, -7.025),   # ~100k hab, depressão sertaneja (Paraíba)
    "Crato_CE":                 (-39.407, -7.233),   # ~130k hab, região metropolitana do Cariri (Ceará)
    "Iguatu_CE":                (-39.299, -6.356),   # ~100k hab, centro-sul cearense (Ceará)

    # Cidades removidas por excesso de nuvens
    # "Feira_de_Santana_BA":    (-38.966, -12.252),  # ~620k hab, entroncamento rodoviário
    # "Campina_Grande_PB":      (-35.880, -7.230),   # ~420k hab, altitude 550 m, polo industrial
    # "Altos":                  (-42.400, -5.000),   # cinturão verde de Teresina
    # "Jose_de_Freitas":        (-42.535, -4.735),   # periurbano de Teresina
    # "Caruaru_PE":             (-36.010, -8.280),   # ~360k hab, altitude 554 m, agreste






}

# ── validação rápida ──────────────────────────────────────────
assert all(len(v) == 2 for v in CIDADES.values()), "Coordenadas inválidas"

# ============================================================
# PARÂMETROS DE DOWNLOAD
# ============================================================
BUFFER_M    = 6_000    # raio do ROI em metros (→ ~12×12 km)
SCALE_M     = 30       # resolução Landsat em metros
MAX_NAN_PCT = 0.30     # descarta frame se >30 % dos pixels forem NaN
MAX_RETRIES = 4        # tentativas por mês antes de desistir
RETRY_WAIT  = 8        # segundos entre tentativas

# ============================================================
# CLOUD MASK
# ============================================================
def mask_landsat_clouds(image):
    qa     = image.select("QA_PIXEL")
    dilated = 1 << 1
    cloud   = 1 << 3
    shadow  = 1 << 4
    # inclui bit de "dilatated cloud" para margens de nuvem
    mask = (qa.bitwiseAnd(dilated).eq(0)
              .And(qa.bitwiseAnd(cloud).eq(0))
              .And(qa.bitwiseAnd(shadow).eq(0)))
    return image.updateMask(mask)

# ============================================================
# ÍNDICES E LST
# ============================================================
dem = ee.Image("USGS/SRTMGL1_003").select("elevation").rename("DEM")

def scale_sr(img, band):
    return img.select(band).multiply(0.0000275).add(-0.2)

def build_month_image(collection, roi):
    med = collection.median()

    b3  = scale_sr(med, "SR_B3")
    b4  = scale_sr(med, "SR_B4")
    b5  = scale_sr(med, "SR_B5")
    b6  = scale_sr(med, "SR_B6")

    ndvi = b5.subtract(b4).divide(b5.add(b4).abs().add(1e-6)).rename("NDVI")
    ndwi = b3.subtract(b5).divide(b3.add(b5).abs().add(1e-6)).rename("NDWI")
    ndbi = b6.subtract(b5).divide(b6.add(b5).abs().add(1e-6)).rename("NDBI")

    # LST em °C — média em vez de mediana para temperatura superficial
    lst = (collection.select("ST_B10")
                     .mean()
                     .multiply(0.00341802)
                     .add(149.0)
                     .subtract(273.15)
                     .rename("LST"))

    # DEM clipado ao mesmo ROI
    dem_clip = dem.clip(roi)

    return ee.Image.cat([ndvi, ndwi, ndbi, dem_clip, lst]).clip(roi)

# ============================================================
# NaN HANDLING
#
# Estratégia: NaN genuíno (nuvem/sombra residual) é preenchido
# por interpolação espacial local via focal_mean do GEE antes
# do download. Se após isso ainda restar >MAX_NAN_PCT, o frame
# é descartado e registrado como gap.
# ============================================================
def fill_nan_spatial(array: np.ndarray) -> np.ndarray:
    """
    Preenche NaN por interpolação dos vizinhos mais próximos
    canal a canal. Mais honesto que substituir por 0.
    Usa scipy se disponível, senão fallback para a média do canal.
    """
    try:
        from scipy.ndimage import generic_filter

        def _fill(channel):
            nan_mask = np.isnan(channel)
            if not nan_mask.any():
                return channel
            filled = channel.copy()
            # média 5×5 ignorando NaN
            def _mean_no_nan(vals):
                v = vals[~np.isnan(vals)]
                return v.mean() if len(v) else np.nan
            filled = generic_filter(filled, _mean_no_nan, size=5, mode="nearest")
            # pixels que ainda são NaN (borda/ilha grande) → média global do canal
            still_nan = np.isnan(filled)
            if still_nan.any():
                ch_mean = np.nanmean(channel)
                filled[still_nan] = ch_mean if not np.isnan(ch_mean) else 0.0
            return filled

        out = np.stack([_fill(array[c]) for c in range(array.shape[0])], axis=0)
        return out

    except ImportError:
        # fallback simples: substituir pela média do canal (melhor que 0)
        out = array.copy()
        for c in range(out.shape[0]):
            ch      = out[c]
            ch_mean = np.nanmean(ch)
            ch[np.isnan(ch)] = ch_mean if not np.isnan(ch_mean) else 0.0
            out[c] = ch
        return out

# ============================================================
# DOWNLOAD COM RETRY
# ============================================================
def _download_with_retry(image, roi, scale, retries=MAX_RETRIES):
    for attempt in range(retries):
        try:
            arr = geemap.ee_to_numpy(image, region=roi, scale=scale)
            return arr
        except Exception as e:
            if attempt < retries - 1:
                wait = RETRY_WAIT * (2 ** attempt)   # backoff exponencial
                print(f"    Tentativa {attempt+1} falhou ({e}). Aguardando {wait}s…")
                time.sleep(wait)
            else:
                raise

# ============================================================
# DOWNLOAD PRINCIPAL
# ============================================================
def download_dataset(start_year: int, end_year: int, folder: str):
    os.makedirs(folder, exist_ok=True)

    landsat8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    landsat9 = ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    base_col = (landsat8.merge(landsat9)
                        .filter(ee.Filter.lt("CLOUD_COVER", 20))
                        .map(mask_landsat_clouds))

    log_ok   = []   # frames salvos
    log_skip = []   # frames descartados (gap)
    log_err  = []   # erros irrecuperáveis

    for city, (lon, lat) in CIDADES.items():
        city_folder = os.path.join(folder, city)
        os.makedirs(city_folder, exist_ok=True)

        roi = ee.Geometry.Point([lon, lat]).buffer(BUFFER_M).bounds()

        for year in range(start_year, end_year + 1):
            for month in range(1, 13):
                start = f"{year}-{month:02d}-01"
                end   = (f"{year+1}-01-01" if month == 12
                         else f"{year}-{month+1:02d}-01")

                col = base_col.filterBounds(roi).filterDate(start, end)

                try:
                    n_images = col.size().getInfo()
                except Exception as e:
                    print(f"  [{city} {year}-{month:02d}] Erro ao contar imagens: {e}")
                    log_err.append({"city": city, "year": year, "month": month, "error": str(e)})
                    continue

                if n_images == 0:
                    log_skip.append({"city": city, "year": year, "month": month,
                                     "reason": "no_images"})
                    continue

                fname = os.path.join(city_folder, f"{year}_{month:02d}.npy")
                if os.path.exists(fname):
                    print(f"  [{city} {year}-{month:02d}] já existe, pulando.")
                    continue

                try:
                    image = build_month_image(col, roi)
                    array = _download_with_retry(image, roi, SCALE_M)

                    if array is None or array.size == 0:
                        log_skip.append({"city": city, "year": year, "month": month,
                                         "reason": "empty_array"})
                        continue

                    # [H, W, C] → [C, H, W]
                    array = np.transpose(array, (2, 0, 1)).astype(np.float32)

                    # ── NaN check ──────────────────────────────────────
                    nan_pct = np.isnan(array).mean()
                    if nan_pct > MAX_NAN_PCT:
                        print(f"  [{city} {year}-{month:02d}] {nan_pct:.0%} NaN — descartado")
                        log_skip.append({"city": city, "year": year, "month": month,
                                         "reason": f"nan_pct={nan_pct:.2f}"})
                        continue

                    # preenchimento espacial dos NaN restantes
                    if np.isnan(array).any():
                        array = fill_nan_spatial(array)

                    np.save(fname, array)
                    log_ok.append({"city": city, "year": year, "month": month,
                                   "n_images": n_images, "shape": list(array.shape)})
                    print(f"  [{city} {year}-{month:02d}] OK  shape={array.shape}  "
                          f"n_imgs={n_images}  nan_orig={nan_pct:.1%}")

                except Exception as e:
                    print(f"  [{city} {year}-{month:02d}] ERRO irrecuperável: {e}")
                    log_err.append({"city": city, "year": year, "month": month, "error": str(e)})

    # ── salva logs ────────────────────────────────────────────────────────
    for name, data in [("log_ok", log_ok), ("log_skip", log_skip), ("log_err", log_err)]:
        with open(os.path.join(folder, f"{name}.json"), "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"\nConcluído: {len(log_ok)} salvos | "
          f"{len(log_skip)} pulados | {len(log_err)} erros")
    print(f"Logs em: {folder}/log_*.json")


# ============================================================
# RESUMO DO DATASET (utilitário pós-download)
# ============================================================
def print_dataset_summary(folder: str):
    """Imprime quantos frames cada cidade tem e aponta gaps."""
    print(f"\n{'Cidade':<30} {'Frames':>6}  {'Gaps (esperados)':>18}")
    print("-" * 60)
    expected = (2024 - 2018 + 1) * 12   # meses entre 2018 e 2024

    for city in sorted(CIDADES):
        city_folder = os.path.join(folder, city)
        if not os.path.isdir(city_folder):
            print(f"  {city:<28} {'—':>6}")
            continue
        frames = len(glob.glob(os.path.join(city_folder, "*.npy")))
        gaps   = expected - frames
        print(f"  {city:<28} {frames:>6}  {gaps:>18}")


if __name__ == "__main__":
    import glob

    FOLDER = f"{FILES_DIR}/heat_dataset_v3"
    download_dataset(start_year=2018, end_year=2024, folder=FOLDER)
    print_dataset_summary(FOLDER)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Treinamento do modelo

In [ ]:
import os
import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import cv2
import functools
from concurrent.futures import ProcessPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIGURAÇÃO GERAL
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

SEQ_LEN = 2 # Quantidade de instantes de tempo no histórico
INPUT_CHANNELS = 8 # NDVI, NDWI, NDBI, DEM, LST_past, Coord_X, Coord_Y, Global_Mean_LST

CH_NDVI, CH_NDWI, CH_NDBI, CH_DEM, CH_LST = 0, 1, 2, 3, 4

BATCH_SIZE = 8 # Reduzido devido à adição da dimensão temporal
ACCUM_STEPS = 2
EPOCHS = 40
LR = 3e-4
WEIGHT_DECAY = 1e-4
K_FOLDS = 5
CROP_SIZE = 256

LST_CLIP_MIN, LST_CLIP_MAX = -20.0, 60.0

def ensure_channels_first(img):
    if img.ndim != 3: raise ValueError(f"Esperado array 3D, recebido {img.shape}")
    if img.shape[0] == 5: return img
    elif img.shape[-1] == 5: return np.transpose(img, (2, 0, 1))
    else: raise ValueError("Não foi possível identificar eixo de canais.")

# ============================================================
# FILTRO DE NUVENS
# ============================================================
def _check_cloud_fraction(filepath, temp_threshold, min_cloud_fraction):
    try:
        data = np.load(filepath)
        lst_band = ensure_channels_first(data)[CH_LST]
        cloud_fraction = np.sum(lst_band < temp_threshold) / lst_band.size
        return Path(filepath), cloud_fraction < min_cloud_fraction
    except Exception:
        return Path(filepath), False

def filter_cloudy_files_runtime(filepaths, temp_threshold=15.0, min_cloud_pct=30.0):
    valid_files = set()
    print(f"\n☁️ Filtrando {len(filepaths)} imagens...")
    check_func = functools.partial(_check_cloud_fraction, temp_threshold=temp_threshold, min_cloud_fraction=min_cloud_pct/100.0)
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(check_func, filepaths))
    for filepath, is_valid in results:
        if is_valid: valid_files.add(filepath)
    print(f" -> Imagens válidas mantidas: {len(valid_files)}\n")
    return valid_files

# ============================================================
# DATASET TEMPORAL COM CORREÇÃO FÍSICA E COORDCONV
# ============================================================
class UrbanHeatTemporalDataset(Dataset):
    def __init__(self, root, allowed_cities, valid_files_set, is_train=False):
        self.root = Path(root)
        self.is_train = is_train
        self.samples = []

        for city in allowed_cities:
            folder = self.root / city
            if not folder.exists(): continue
            files = sorted([f for f in folder.glob("*.npy") if f in valid_files_set])

            # Necessário SEQ_LEN imagens para entrada + 1 para o alvo
            if len(files) < SEQ_LEN + 1: continue

            for i in range(len(files) - SEQ_LEN):
                # (Lista de entradas, Arquivo Alvo)
                seq_files = filecvs[i : i + SEQ_LEN]
                target_file = files[i + SEQ_LEN]
                self.samples.append((seq_files, target_file))

        # Criação dos grids de coordenadas para o CoordConv
        y_grid, x_grid = np.mgrid[-1:1:complex(0, CROP_SIZE), -1:1:complex(0, CROP_SIZE)]
        self.coord_y = y_grid.astype(np.float32)
        self.coord_x = x_grid.astype(np.float32)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        seq_files, target_file = self.samples[idx]

        # 1. Carrega as imagens completas
        seq_imgs = [ensure_channels_first(np.load(f).astype(np.float32)) for f in seq_files]
        tgt_img = ensure_channels_first(np.load(target_file).astype(np.float32))

        # 2. Definição do Crop Central ou Aleatório
        max_h, max_w = tgt_img.shape[1], tgt_img.shape[2]
        if self.is_train and max_h > CROP_SIZE and max_w > CROP_SIZE:
            y, x = random.randint(0, max_h - CROP_SIZE), random.randint(0, max_w - CROP_SIZE)
        else:
            y, x = max(0, (max_h - CROP_SIZE) // 2), max(0, (max_w - CROP_SIZE) // 2)

        # 3. Tratamento do Alvo (Anomalia baseada na CIDADE INTEIRA)
        is_nodata_tgt = (tgt_img[CH_NDVI] == 0.0) & (tgt_img[CH_NDBI] == 0.0)
        valid_mask_tgt_full = ~is_nodata_tgt

        valid_pixels_tgt = tgt_img[CH_LST][valid_mask_tgt_full]
        city_mean_target = valid_pixels_tgt.mean() if len(valid_pixels_tgt) > 0 else 0.0

        # Anomalia Global calculada antes do crop
        tgt_anomaly_full = (tgt_img[CH_LST] - city_mean_target) * valid_mask_tgt_full.astype(np.float32)

        cropped_tgt_anomaly = tgt_anomaly_full[y:y+CROP_SIZE, x:x+CROP_SIZE]
        crop_valid_mask = valid_mask_tgt_full[y:y+CROP_SIZE, x:x+CROP_SIZE].astype(np.float32)

        padded_tgt_anomaly = np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.float32)
        valid_mask = np.zeros((1, CROP_SIZE, CROP_SIZE), dtype=np.float32)
        padded_tgt_anomaly[:cropped_tgt_anomaly.shape[0], :cropped_tgt_anomaly.shape[1]] = cropped_tgt_anomaly
        valid_mask[0, :crop_valid_mask.shape[0], :crop_valid_mask.shape[1]] = crop_valid_mask

        target_norm = (np.clip(padded_tgt_anomaly, -20.0, 20.0) / 5.0) * valid_mask[0]

        # 4. Construção da Sequência (Histórico)
        seq_tensors = []
        for img_t in seq_imgs:
            # Calcula a média da cidade no instante t
            mask_t = ~((img_t[CH_NDVI] == 0.0) & (img_t[CH_NDBI] == 0.0))
            city_mean_t = img_t[CH_LST][mask_t].mean() if mask_t.sum() > 0 else 0.0
            anomaly_t = (img_t[CH_LST] - city_mean_t) * mask_t.astype(np.float32)

            cropped_base = img_t[:, y:y+CROP_SIZE, x:x+CROP_SIZE]
            cropped_anomaly_t = anomaly_t[y:y+CROP_SIZE, x:x+CROP_SIZE]

            padded_base = np.zeros((5, CROP_SIZE, CROP_SIZE), dtype=np.float32)
            padded_base[:, :cropped_base.shape[1], :cropped_base.shape[2]] = cropped_base

            padded_base[CH_NDVI] = np.clip(padded_base[CH_NDVI], -1.0, 1.0)
            padded_base[CH_NDWI] = np.clip(padded_base[CH_NDWI], -1.0, 1.0)
            padded_base[CH_NDBI] = np.clip(padded_base[CH_NDBI], -1.0, 1.0)

            if valid_mask.sum() > 0:
                local_dem_mean = padded_base[CH_DEM][valid_mask[0] == 1.0].mean()
                padded_base[CH_DEM] = (padded_base[CH_DEM] - local_dem_mean) / 50.0

            # Anomalia LST do passado normalizada
            lst_past_norm = np.clip(cropped_anomaly_t, -20.0, 20.0) / 5.0
            lst_past_padded = np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.float32)
            lst_past_padded[:lst_past_norm.shape[0], :lst_past_norm.shape[1]] = lst_past_norm

            # Global Mean (Proxy Meteorológico) expandido espacialmente
            global_mean_layer = np.full((CROP_SIZE, CROP_SIZE), city_mean_t / 40.0, dtype=np.float32)

            # Empilha: [NDVI, NDWI, NDBI, DEM, LST_past_anomaly, CoordY, CoordX, GlobalMean]
            step_features = np.stack([
                padded_base[CH_NDVI] * valid_mask[0],
                padded_base[CH_NDWI] * valid_mask[0],
                padded_base[CH_NDBI] * valid_mask[0],
                padded_base[CH_DEM] * valid_mask[0],
                lst_past_padded * valid_mask[0],
                self.coord_y * valid_mask[0],
                self.coord_x * valid_mask[0],
                global_mean_layer * valid_mask[0]
            ], axis=0)

            seq_tensors.append(torch.from_numpy(step_features).float())

        # seq_out shape: (Time, Channels, H, W)
        seq_out = torch.stack(seq_tensors, dim=0)
        out_tgt = torch.from_numpy(target_norm).unsqueeze(0).float()
        out_valid = torch.from_numpy(valid_mask).float()

        # Augmentation (aplicado consistentemente em todo o volume de tempo)
        if self.is_train:
            if random.random() > 0.5:
                seq_out = torch.flip(seq_out, dims=[-1])
                out_tgt, out_valid = torch.flip(out_tgt, dims=[-1]), torch.flip(out_valid, dims=[-1])
            if random.random() > 0.5:
                seq_out = torch.flip(seq_out, dims=[-2])
                out_tgt, out_valid = torch.flip(out_tgt, dims=[-2]), torch.flip(out_valid, dims=[-2])
            k = random.randint(0, 3)
            if k > 0:
                seq_out = torch.rot90(seq_out, k, dims=[-2, -1])
                out_tgt, out_valid = torch.rot90(out_tgt, k, dims=[-2, -1]), torch.rot90(out_valid, k, dims=[-2, -1])

        return seq_out, out_tgt, out_valid, city_mean_target

# ============================================================
# ARQUITETURA U-NET COM CONV-LSTM NO GARGALO
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(in_channels=input_dim + hidden_dim,
                              out_channels=4 * hidden_dim,
                              kernel_size=kernel_size,
                              padding=padding, bias=True)

    def forward(self, input_tensor, cur_state):
        h_cur, c_cur = cur_state
        combined = torch.cat([input_tensor, h_cur], dim=1)
        combined_conv = self.conv(combined)

        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)
        i = torch.sigmoid(cc_i)
        f = torch.sigmoid(cc_f)
        o = torch.sigmoid(cc_o)
        g = torch.tanh(cc_g)

        c_next = f * c_cur + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next

class UNetConvLSTM(nn.Module):
    def __init__(self, in_channels=8, features=[32, 64, 128]):
        super().__init__()
        self.features = features
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder Espacial (aplica-se a cada frame de tempo)
        curr_in = in_channels
        for feature in features:
            self.downs.append(DoubleConv(curr_in, feature))
            curr_in = feature

        # Bottleneck Temporal
        self.convlstm = ConvLSTMCell(input_dim=features[-1], hidden_dim=features[-1]*2)

        # Decoder Espacial
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], 1, kernel_size=1)

    def forward(self, x):
        # x shape: (B, T, C, H, W)
        b, t, c, h, w = x.shape

        # O Encoder processa todos os passos de tempo simultaneamente no batch
        x_reshaped = x.view(b * t, c, h, w)

        skip_connections = []
        for down in self.downs:
            x_reshaped = down(x_reshaped)
            skip_connections.append(x_reshaped)
            x_reshaped = self.pool(x_reshaped)

        # Prepara para ConvLSTM
        _, c_feat, h_feat, w_feat = x_reshaped.shape
        x_seq = x_reshaped.view(b, t, c_feat, h_feat, w_feat)

        h_state = torch.zeros(b, c_feat * 2, h_feat, w_feat, device=x.device)
        c_state = torch.zeros(b, c_feat * 2, h_feat, w_feat, device=x.device)

        # Loop temporal no Bottleneck
        for time_step in range(t):
            h_state, c_state = self.convlstm(x_seq[:, time_step, :, :, :], (h_state, c_state))

        out = h_state # Pega apenas a memória final do último instante

        # No Decoder, as skip connections usam apenas os tensores do ÚLTIMO passo de tempo (t-1)
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            out = self.ups[i](out)
            skip = skip_connections[i//2].view(b, t, -1, out.shape[2], out.shape[3])[:, -1, :, :, :]

            if out.shape != skip.shape:
                out = F.interpolate(out, size=skip.shape[2:], mode='bilinear', align_corners=False)

            concat_skip = torch.cat((skip, out), dim=1)
            out = self.ups[i+1](concat_skip)

        return self.final_conv(out)

# ============================================================
# MÉTRICAS CORRIGIDAS (Reconstrução com City Mean)
# ============================================================
def compute_metrics(pred_norm, target_norm, valid_mask, city_mean, top_q=0.9):
    # A reconstrução para Celsius depende da média global da cidade!
    # pred_c = (anomalia_prevista * 5.0) + media_da_cidade

    # Se city_mean for um tensor (Batch,), remodelamos para suportar broadcast
    if isinstance(city_mean, torch.Tensor):
        city_mean = city_mean.view(-1, 1, 1, 1)

    pred_c = (pred_norm * 5.0) + city_mean
    target_c = (target_norm * 5.0) + city_mean

    valid_count = valid_mask.sum() + 1e-8
    mae_celsius = (torch.abs(pred_c - target_c) * valid_mask).sum() / valid_count

    intersection, union, hotspot_mae, hs_count = 0.0, 0.0, 0.0, 0.0
    for i in range(target_norm.size(0)):
        valid_pixels_real = target_c[i][valid_mask[i] == 1.0]
        valid_pixels_pred = pred_c[i][valid_mask[i] == 1.0]

        if valid_pixels_real.numel() > 0:
            q90_real = torch.quantile(valid_pixels_real, top_q)
            q90_pred = torch.quantile(valid_pixels_pred, top_q)

            hs_real_mask = (target_c[i] >= q90_real) & (valid_mask[i] == 1.0)
            hs_pred_mask = (pred_c[i] >= q90_pred) & (valid_mask[i] == 1.0)

            intersection += (hs_pred_mask & hs_real_mask).float().sum()
            union += (hs_pred_mask | hs_real_mask).float().sum()

            if hs_real_mask.sum() > 0:
                hotspot_mae += (torch.abs(pred_c[i] - target_c[i]) * hs_real_mask.float()).sum()
                hs_count += hs_real_mask.float().sum()

    return mae_celsius.item(), (hotspot_mae / (hs_count + 1e-8)).item(), ((intersection + 1e-6) / (union + 1e-6)).item()

class BaselineLoss(nn.Module):
    def __init__(self, hotspot_w=2.0, top_q=0.9):
        super().__init__()
        self.hotspot_w = hotspot_w
        self.top_q = top_q

    def forward(self, pred, target, valid_mask):
        pred = pred * valid_mask
        target = target * valid_mask

        l1_diff = torch.abs(pred - target)
        valid_count = valid_mask.sum() + 1e-8
        base_loss = l1_diff.sum() / valid_count

        hotspot_mask = torch.zeros_like(target)
        for i in range(target.size(0)):
            valid_pixels = target[i][valid_mask[i] == 1.0]
            if valid_pixels.numel() > 0:
                q90 = torch.quantile(valid_pixels, self.top_q)
                hotspot_mask[i] = (target[i] >= q90).float() * valid_mask[i]

        hs_count = hotspot_mask.sum() + 1e-8
        hotspot_loss = (l1_diff * hotspot_mask).sum() / hs_count if hotspot_mask.sum() > 0 else torch.tensor(0.0, device=target.device)

        return base_loss + (self.hotspot_w * hotspot_loss)

# ============================================================
# INTERPRETABILIDADE: GRAD-CAM PARA CONV-LSTM
# ============================================================
class RegressionGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(self, input_tensor):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output.sum()
        score.backward()

        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().detach().numpy()

        if cam.ndim == 3: cam = cam[0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output.squeeze().cpu().detach().numpy()

def plot_grad_cam(input_tensor, cam_map, pred_norm, real_norm, city_mean, save_path="grad_cam.png"):
    # Pegamos o NDVI do último instante de tempo do histórico (t-1)
    ndvi_input = input_tensor[0, -1, CH_NDVI].cpu().numpy()

    # Reconstrução LST Absoluta usando a média correta do Batch/Crop
    pred_celsius = (pred_norm * 5.0) + city_mean
    real_celsius = (real_norm * 5.0) + city_mean

    if pred_celsius.ndim == 3: pred_celsius = pred_celsius[0]
    if real_celsius.ndim == 3: real_celsius = real_celsius[0]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(ndvi_input, cmap='gray')
    axes[0].set_title('NDVI (Instante Atual)')

    heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    axes[1].imshow(ndvi_input, cmap='gray', alpha=0.5)
    axes[1].imshow(heatmap, alpha=0.6)
    axes[1].set_title('Atenção Espaço-Temporal')

    im2 = axes[2].imshow(pred_celsius, cmap='inferno')
    axes[2].set_title('Temp. Prevista (°C)')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    im3 = axes[3].imshow(real_celsius, cmap='inferno')
    axes[3].set_title('Temp. Real (°C)')
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================
if __name__ == "__main__":
    all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]
    all_files_raw = list(Path(DATA_DIR).rglob("*.npy"))
    valid_files_set = filter_cloudy_files_runtime(all_files_raw)

    print("\n2. Iniciando K-Fold Cross Validation Espaço-Temporal...")
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    folds_list = list(kf.split(all_cities))

    for fold, (train_idx, val_idx) in enumerate(folds_list):
        train_cities = [all_cities[i] for i in train_idx]
        val_cities = [all_cities[i] for i in val_idx]

        train_ds = UrbanHeatTemporalDataset(DATA_DIR, allowed_cities=train_cities, valid_files_set=valid_files_set, is_train=True)
        val_ds = UrbanHeatTemporalDataset(DATA_DIR, allowed_cities=val_cities, valid_files_set=valid_files_set, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = UNetConvLSTM(in_channels=INPUT_CHANNELS).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=6)

        loss_fn = BaselineLoss()
        best_loss = float('inf')

        print(f"\n--- Iniciando FOLD {fold + 1} ---")
        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0
            optimizer.zero_grad()

            for step, (seq, target, valid, city_mean) in enumerate(train_loader):
                seq, target, valid = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE)

                pred = model(seq)
                loss = loss_fn(pred, target, valid)
                (loss / ACCUM_STEPS).backward()
                train_loss += loss.item()

                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

            model.eval()
            val_loss, val_mae_c, val_hs_mae_c, val_iou = 0, 0, 0, 0

            with torch.no_grad():
                for seq, target, valid, city_mean in val_loader:
                    seq, target, valid = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE)
                    pred = model(seq)

                    val_loss += loss_fn(pred, target, valid).item()

                    # Passando o city_mean do batch para reconstrução térmica correta
                    mae_c, hs_mae_c, iou = compute_metrics(pred, target, valid, city_mean.to(DEVICE), top_q=0.9)
                    val_mae_c += mae_c
                    val_hs_mae_c += hs_mae_c
                    val_iou += iou

            n_t, n_v = max(len(train_loader), 1), max(len(val_loader), 1)
            train_loss /= n_t
            val_loss /= n_v
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1:02d} | T.Loss: {train_loss:.4f} | V.Loss: {val_loss:.4f} | MAE (°C): {val_mae_c/n_v:.4f} | HS MAE: {val_hs_mae_c/n_v:.4f} | IoU: {val_iou/n_v:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(model.state_dict(), f"best_convlstm_fold_{fold}.pt")

        # =======================================================
        # GERAÇÃO DO GRAD-CAM NO FIM DO FOLD
        # =======================================================
        print(f"\nGerando Grad-CAM Espaço-Temporal para o Fold {fold+1}...")

        model.load_state_dict(torch.load(f"best_convlstm_fold_{fold}.pt", weights_only=True))
        cam_generator = RegressionGradCAM(model, target_layer=model.ups[-1])

        sample_seq, sample_tgt, sample_valid, sample_city_mean = next(iter(val_loader))

        cam_map, pred_np = cam_generator.generate(sample_seq[0:1].to(DEVICE))
        real_np = sample_tgt[0:1].cpu().numpy()
        city_mean_val = sample_city_mean[0].item()

        plot_grad_cam(
            sample_seq[0:1],
            cam_map,
            pred_np,
            real_np,
            city_mean_val,
            save_path=f"gradcam_convlstm_fold_{fold+1}.png"
        )
        print(f"-> Imagem do Grad-CAM salva como 'gradcam_convlstm_fold_{fold+1}.png'\n")

In [ ]:
import os
import shutil
import random
import numpy as np
from pathlib import Path
import functools
from concurrent.futures import ProcessPoolExecutor

# ============================================================
# CONFIGURAÇÃO GERAL
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("/content/drive/MyDrive/IA/Pibic/Vulnerability/heat_dataset_v3")
# Pasta de destino para onde as imagens poluídas serão movidas
QUARANTINE_DIR = Path("/content/drive/MyDrive/IA/Pibic/Vulnerability/heat_dataset_v3_poluidas")

CH_LST = 4

def ensure_channels_first(img):
    if img.ndim != 3: raise ValueError(f"Esperado array 3D, recebido {img.shape}")
    if img.shape[0] == 5: return img
    elif img.shape[-1] == 5: return np.transpose(img, (2, 0, 1))
    else: raise ValueError("Não foi possível identificar eixo de canais.")

# ============================================================
# FILTRO E MOVIMENTAÇÃO DE NUVENS
# ============================================================
def _check_and_classify_file(filepath, temp_threshold, min_cloud_fraction):
    """
    Verifica a fração de nuvens e retorna o caminho do ficheiro juntamente com
    um booleano indicando se é válido (True) ou se deve ser movido (False).
    """
    try:
        data = np.load(filepath)
        lst_band = ensure_channels_first(data)[CH_LST]
        cloud_fraction = np.sum(lst_band < temp_threshold) / lst_band.size
        is_valid = cloud_fraction < min_cloud_fraction
        return Path(filepath), is_valid
    except Exception:
        # Se houver erro ao ler o ficheiro, considera inválido/corrompido por precaução
        return Path(filepath), False

def move_cloudy_files_recursive(temp_threshold=15.0, min_cloud_pct=30.0):
    QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)

    # Procura recursivamente por todos os ficheiros .npy em todas as subpastas (cidades)
    all_files = list(DATA_DIR.rglob("*.npy"))

    if not all_files:
        print(f"Nenhum ficheiro .npy encontrado em {DATA_DIR} ou nas suas subpastas.")
        return

    valid_count = 0
    moved_count = 0

    print(f"\n☁️ A analisar {len(all_files)} imagens distribuídas pelas pastas de cidades...")

    check_func = functools.partial(
        _check_and_classify_file,
        temp_threshold=temp_threshold,
        min_cloud_fraction=min_cloud_pct/100.0
    )

    # Processamento paralelo para agilizar a verificação
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(check_func, all_files))

    for filepath, is_valid in results:
        if is_valid:
            valid_count += 1
        else:
            try:
                # Opcional: Se quiser manter o nome da cidade no nome do ficheiro ao mover
                # (ex: teresina_imagem123.npy) para evitar conflitos caso cidades diferentes
                # tenham ficheiros com exatamente o mesmo nome:
                relative_path = filepath.relative_to(DATA_DIR)
                dest_filename = f"{str(relative_path).replace(os.sep, '_')}"
                dest_path = QUARANTINE_DIR / dest_filename

                shutil.move(str(filepath), str(dest_path))
                moved_count += 1
            except Exception as e:
                print(f"Erro ao mover o ficheiro {filepath.name}: {e}")

    print(f" -> Imagens válidas mantidas nas pastas de origem: {valid_count}")
    print(f" -> Imagens poluídas movidas para '{QUARANTINE_DIR}': {moved_count}\n")

# ============================================================
# EXECUÇÃO
# ============================================================
if __name__ == "__main__":
    move_cloudy_files_recursive(
        temp_threshold=15.0,
        min_cloud_pct=30.0
    )

In [ ]:
import os
import random
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn.functional as F

# ============================================================
# 1. SIMULAÇÃO DE INTERVENÇÃO COM ORÇAMENTO PARAMETRIZADO
# ============================================================
def run_intervention_simulation(model, input_tensor, valid_mask, city_mean, scale_factor=5.0, budget=30, known_hs_mae=1.5):
    """
    input_tensor: shape (1, Time, Channels, H, W)
    valid_mask: shape (1, 1, H, W)
    city_mean: float ou tensor escalar (temperatura média da cidade)
    """
    model.eval()
    CH_NDVI, CH_NDWI, CH_NDBI, CH_DEM = 0, 1, 2, 3

    # --- CENÁRIO BASE ---
    input_tensor = input_tensor.clone().detach()
    input_tensor.requires_grad_(True)

    # Forward Pass
    base_pred_norm = model(input_tensor)

    # Reconstrução Térmica Corrigida (usando city_mean)
    base_celsius = (base_pred_norm * scale_factor) + city_mean
    base_celsius_np = base_celsius.squeeze().cpu().detach().numpy()

    # Extrai o estado atual (último instante de tempo: index -1)
    current_ndvi = input_tensor[0, -1, CH_NDVI, :, :].detach()
    current_ndwi = input_tensor[0, -1, CH_NDWI, :, :].detach()
    current_dem  = input_tensor[0, -1, CH_DEM, :, :].detach()

    # --- Filtros Restritivos (Solo/Concreto, Sem Água, Sem Relevo Íngreme) ---
    is_concreto = (current_ndvi < 0.2).float()
    not_water = (current_ndwi < 0.1).float()
    dem_mean = current_dem[valid_mask[0, 0] == 1.0].mean()
    is_flat = (torch.abs(current_dem - dem_mean) < 1.5).float()

    valid_intervention_base = is_concreto * not_water * is_flat * valid_mask[0, 0].to(input_tensor.device)
    valid_base_np = valid_intervention_base.cpu().numpy()

    valid_y, valid_x = np.where(valid_base_np == 1)
    if len(valid_y) < budget:
        print(f"⚠️ Área válida insuficiente para o budget solicitado ({budget} pixels).")
        return None

    # --- CÁLCULO DO GRADIENTE TÉRMICO ---
    valid_pixels = base_celsius[valid_mask == 1.0]
    q90_temp = torch.quantile(valid_pixels, 0.9)
    hotspot_mask = (base_celsius >= q90_temp).float() * valid_mask
    total_hotspot_heat = (base_celsius * hotspot_mask).sum()

    model.zero_grad()
    total_hotspot_heat.backward()

    # 1. Gradiente Composto (Sensibilidade a TODAS as variáveis alteradas)
    # Pegamos os gradientes do último instante temporal (-1)
    grad_ndvi = -input_tensor.grad[0, -1, CH_NDVI, :, :].cpu().numpy()
    grad_ndwi = -input_tensor.grad[0, -1, CH_NDWI, :, :].cpu().numpy()
    grad_ndbi = input_tensor.grad[0, -1, CH_NDBI, :, :].cpu().numpy() # Invertido pois queremos reduzir o NDBI
    grad_lst_past = input_tensor.grad[0, -1, 4, :, :].cpu().numpy()   # Canal 4 (Inércia térmica)

    # 2. Soma Ponderada do Potencial de Mitigação
    # Focamos apenas nas direções que ajudam a esfriar (ReLU)
    composite_gradient = (
        np.maximum(0, grad_ndvi) * 0.3 +
        np.maximum(0, grad_ndwi) * 0.2 +
        np.maximum(0, grad_ndbi) * 0.2 +
        np.maximum(0, grad_lst_past) * 1.0  # A inércia tem peso maior
    )

    # 3. Suavização Espacial (Filtro de Ruído)
    # Remove "artefatos matemáticos" de 1 pixel e força a busca por clusters férteis
    smoothed_gradient = cv2.GaussianBlur(composite_gradient, (5, 5), 1.0)

    # ================= ESTRATÉGIAS DE ALOCAÇÃO =================

    # A. MULTI-CRITÉRIO HÍBRIDO (Gradiente + Urgência Térmica)
    # Normalizamos o gradiente e a temperatura para somá-los de forma justa
    grad_norm = (smoothed_gradient - smoothed_gradient.min()) / (smoothed_gradient.max() - smoothed_gradient.min() + 1e-8)
    temp_norm = (base_celsius_np - base_celsius_np.min()) / (base_celsius_np.max() - base_celsius_np.min() + 1e-8)

    # Score final: 60% potencial de esfriamento (Rede Neural) + 40% urgência (Física)
    hybrid_score = (grad_norm * 0.6) + (temp_norm * 0.4)
    planting_potential = hybrid_score * valid_base_np

    gradient_vals = planting_potential[valid_y, valid_x]
    top_multi_idx = np.argsort(gradient_vals)[::-1][:budget]
    mask_multi = np.zeros_like(base_celsius_np, dtype=bool)
    mask_multi[valid_y[top_multi_idx], valid_x[top_multi_idx]] = True

    # B. GULOSA
    valid_temps = base_celsius_np[valid_y, valid_x]
    top_greedy_idx = np.argsort(valid_temps)[::-1][:budget]
    mask_greedy = np.zeros_like(base_celsius_np, dtype=bool)
    mask_greedy[valid_y[top_greedy_idx], valid_x[top_greedy_idx]] = True

    # C. ALEATÓRIA
    random_idx = np.random.choice(len(valid_y), size=budget, replace=False)
    mask_random = np.zeros_like(base_celsius_np, dtype=bool)
    mask_random[valid_y[random_idx], valid_x[random_idx]] = True

    # ================= FUNÇÃO AVALIADORA TERMODINÂMICA =================
    def evaluate_intervention(mask):
        simulated_input = input_tensor.clone().detach()

        # 1. Modifica a cobertura do solo em toda a sequência de tempo
        simulated_input[0, :, CH_NDVI, mask] = 0.55
        simulated_input[0, :, CH_NDBI, mask] = -0.15
        simulated_input[0, :, CH_NDWI, mask] = 0.10

        # 2. CORREÇÃO DA INÉRCIA TÉRMICA (Canal 4 = LST_past)
        # Definimos como -0.2 (representando que a área já era 1.0°C mais fria que a média da cidade)
        simulated_input[0, :, 4, mask] = -0.20

        with torch.no_grad():
            future_celsius = (model(simulated_input) * scale_factor) + city_mean

        raw_cooling = (base_celsius - future_celsius).squeeze().cpu().detach().numpy()

        # Advecção Multi-escala (Simula dispersão do vento)
        local_cooling = cv2.GaussianBlur(raw_cooling, (5, 5), 1.0)
        breeze_cooling = cv2.GaussianBlur(raw_cooling, (15, 15), 3.0)
        smoothed_cooling = (0.7 * local_cooling) + (0.3 * breeze_cooling)

        avg_cooling = smoothed_cooling[mask].mean() if mask.sum() > 0 else 0
        return smoothed_cooling, avg_cooling

    cool_multi, avg_multi = evaluate_intervention(mask_multi)
    cool_greedy, avg_greedy = evaluate_intervention(mask_greedy)
    cool_rand, avg_rand = evaluate_intervention(mask_random)

    return {
        'base_celsius': base_celsius_np,
        'error_margin': known_hs_mae,
        'budget': budget,
        'multi': {'mask': mask_multi, 'cooling': cool_multi, 'avg_cooling': avg_multi},
        'greedy': {'mask': mask_greedy, 'cooling': cool_greedy, 'avg_cooling': avg_greedy},
        'random': {'mask': mask_random, 'cooling': cool_rand, 'avg_cooling': avg_rand}
    }


# ============================================================
# 2. PLOTAGEM ESPACIAL COMPACTA (PARA O ARTIGO)
# ============================================================
def plot_intervention_compact(results, save_path="figura_artigo_espacial.png"):
    if results is None: return

    fig, axes = plt.subplots(2, 4, figsize=(12, 5.5), gridspec_kw={'wspace': 0.05, 'hspace': 0.1})

    for ax in axes.flat:
        ax.set_xticks([]); ax.set_yticks([])

    vmin, vmax = results['base_celsius'].min(), results['base_celsius'].max()

    # Base e Título
    im_base = axes[0, 0].imshow(results['base_celsius'], cmap='coolwarm', vmin=vmin, vmax=vmax)
    axes[0, 0].set_title('LST Base (°C)', fontsize=10, fontweight='bold')

    strategies = [
        ('Multi-Critério', results['multi'], axes[0, 1], axes[1, 1]),
        ('Gulosa', results['greedy'], axes[0, 2], axes[1, 2]),
        ('Aleatória', results['random'], axes[0, 3], axes[1, 3])
    ]

    for title, data, ax_mask, ax_cool in strategies:
        # Fundo cinza + pixels de intervenção
        ax_mask.imshow(results['base_celsius'], cmap='gray', alpha=0.6)
        overlay = np.zeros((*data['mask'].shape, 4))
        overlay[data['mask']] = [0, 1, 0, 1.0] # Verde forte para contraste
        ax_mask.imshow(overlay)
        ax_mask.set_title(f'Alocação: {title}', fontsize=10)

        # Mapas de Resfriamento
        im_cool = ax_cool.imshow(data['cooling'], cmap='Blues', vmin=0, vmax=results['multi']['cooling'].max())
        ax_cool.set_title(f'ΔT Resfriamento', fontsize=10)

    # Bloco de texto com métricas
    axes[1, 0].axis('off')
    stats_text = (
        f"Métricas (Budget={results['budget']} px)\n"
        f"------------------------\n"
        f"Redução Média (ΔT):\n"
        f"• Multi: -{results['multi']['avg_cooling']:.2f} °C\n"
        f"• Gulosa: -{results['greedy']['avg_cooling']:.2f} °C\n"
        f"• Aleat.: -{results['random']['avg_cooling']:.2f} °C\n\n"
        f"MAE Modelo: ±{results['error_margin']} °C"
    )
    axes[1, 0].text(0.5, 0.5, stats_text, ha='center', va='center', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray'))

    # Colorbars ajustados
    cbar_ax_base = fig.add_axes([0.13, 0.52, 0.01, 0.35])
    fig.colorbar(im_base, cax=cbar_ax_base)

    cbar_ax_cool = fig.add_axes([0.91, 0.12, 0.01, 0.35])
    fig.colorbar(im_cool, cax=cbar_ax_cool, label='ΔT (°C)')

    plt.subplots_adjust(left=0.18, right=0.9, top=0.9, bottom=0.1)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ============================================================
# 3. PLOTAGEM DA CURVA DE SENSIBILIDADE DO BUDGET
# ============================================================
def plot_budget_sensitivity_curve(model, input_tensor, valid_mask, city_mean, scale_factor, budgets, save_path="figura_artigo_sensibilidade.png"):
    print("\n📈 Executando análise paramétrica (Sensibilidade de Orçamento)...")
    res_multi, res_greedy, res_random = [], [], []
    valid_budgets = []

    for b in budgets:
        res = run_intervention_simulation(model, input_tensor, valid_mask, city_mean, scale_factor, budget=b)
        if res is not None:
            res_multi.append(res['multi']['avg_cooling'])
            res_greedy.append(res['greedy']['avg_cooling'])
            res_random.append(res['random']['avg_cooling'])
            valid_budgets.append(b)
            print(f"   - Budget {b:3d}: Multi (-{res['multi']['avg_cooling']:.2f}°C) | Gulosa (-{res['greedy']['avg_cooling']:.2f}°C)")

    if not valid_budgets:
        print("❌ Nenhum budget válido para plotar.")
        return

    plt.figure(figsize=(8, 5))
    plt.plot(valid_budgets, res_multi, marker='o', linestyle='-', color='#10b981', linewidth=2.5, label='Multi-Critério (Proposto)')
    plt.plot(valid_budgets, res_greedy, marker='s', linestyle='--', color='#ef4444', linewidth=2, label='Baseline Gulosa')
    plt.plot(valid_budgets, res_random, marker='^', linestyle=':', color='#6b7280', linewidth=2, label='Baseline Aleatória')
    plt.title('Eficiência de Resfriamento vs. Orçamento de Intervenção', fontsize=12, fontweight='bold', pad=12)
    plt.xlabel('Orçamento de Intervenção (Número de Pixels, ~900m² cada)', fontsize=10)
    plt.ylabel('Redução Média de LST Absoluta (|ΔT| °C)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True, facecolor='white', edgecolor='gray')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()


# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================
if __name__ == "__main__":
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    SCALE_FACTOR = 5.0
    INDICE_IMAGEM = 0
    BUDGET_ESPACIAL = 45
    BUDGETS_CURVA = [10, 20, 30, 40, 50, 75, 100, 150]
    DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

    # 1. Carrega o modelo novo (ConvLSTM)
    # OBS: Certifique-se de que a classe UNetConvLSTM já foi executada na célula anterior do seu Jupyter Notebook
    model = UNetConvLSTM(in_channels=8).to(DEVICE)
    modelo_path = "best_convlstm_fold_0.pt"

    if os.path.exists(modelo_path):
        model.load_state_dict(torch.load(modelo_path, map_location=DEVICE, weights_only=True))
    else:
        print(f"⚠️ AVISO: Pesos não encontrados ({modelo_path}). Usando pesos aleatórios.")
    model.eval()

    # 2. Carrega os dados com o Dataset Temporal
    all_files_raw = list(Path(DATA_DIR).rglob("*.npy"))
    valid_files_set = filter_cloudy_files_runtime(all_files_raw)

    teresina_ds = UrbanHeatTemporalDataset(root=DATA_DIR, allowed_cities=["Teresina"], valid_files_set=valid_files_set, is_train=False)

    if len(teresina_ds) == 0:
        print("❌ Nenhum dado encontrado para Teresina.")
    else:
        # Acesso Direto pelo Índice
        sample_seq, sample_tgt, sample_valid, city_mean = teresina_ds[INDICE_IMAGEM]

        # Adicionando a dimensão de batch
        input_tensor = sample_seq.unsqueeze(0).to(DEVICE)
        valid_mask = sample_valid.unsqueeze(0).to(DEVICE)

        # Formata o city_mean para escalar Python (ou tensor no mesmo device)
        if isinstance(city_mean, torch.Tensor):
            city_mean = city_mean.to(DEVICE).item()

        print(f"\n🌳 INICIANDO SIMULAÇÃO DE INTERVENÇÃO (Imagem Índice: {INDICE_IMAGEM})")
        print("="*70)

        res_espacial = run_intervention_simulation(
            model=model,
            input_tensor=input_tensor,
            valid_mask=valid_mask,
            city_mean=city_mean,
            scale_factor=SCALE_FACTOR,
            budget=BUDGET_ESPACIAL,
            known_hs_mae=1.5
        )

        if res_espacial:
            plot_intervention_compact(res_espacial, "figura_artigo_espacial.png")
            print(f"✅ Mapa espacial gerado (Budget {BUDGET_ESPACIAL}): 'figura_artigo_espacial.png'")

        plot_budget_sensitivity_curve(
            model=model,
            input_tensor=input_tensor,
            valid_mask=valid_mask,
            city_mean=city_mean,
            scale_factor=SCALE_FACTOR,
            budgets=BUDGETS_CURVA,
            save_path="figura_artigo_sensibilidade.png"
        )

        print("\n🎉 Todas as simulações foram concluídas com sucesso!")

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np

TOTAL_RAW_CHANNELS = 5

def ensure_channels_first(img):
    if img.ndim != 3: raise ValueError(f"Esperado array 3D, recebido {img.shape}")
    if img.shape[0] == TOTAL_RAW_CHANNELS: return img
    elif img.shape[-1] == TOTAL_RAW_CHANNELS: return np.transpose(img, (2, 0, 1))
    else: raise ValueError("Não foi possível identificar eixo de canais.")


def analyze_cloudy_cities(data_dir, temp_threshold=15.0, min_cloud_pct=30.0):
    """
    Analisa o dataset, identifica as fotos nubladas por cidade
    e exibe um ranking ordenado da cidade com mais fotos nubladas para a com menos.
    """
    root_path = Path(data_dir)
    min_cloud_fraction = min_cloud_pct / 100.0

    # Dicionário para armazenar as contagens por cidade
    cloudy_stats = defaultdict(lambda: {"total": 0, "cloudy": 0})

    print(f"\n🔍 Analisando nuvens por cidade (Limite: >{min_cloud_pct}%)...")

    # Percorre todas as pastas de cidades
    for city_folder in root_path.iterdir():
        if not city_folder.is_dir() or city_folder.name.startswith('.'):
            continue

        city_name = city_folder.name
        npy_files = list(city_folder.glob("*.npy"))

        cloudy_stats[city_name]["total"] = len(npy_files)

        for filepath in npy_files:
            try:
                data = np.load(filepath)
                # Utiliza a mesma função de extração de canais já definida no seu script
                lst_band = ensure_channels_first(data)[CH_LST]

                # Calcula a fração de pixels abaixo do limiar de temperatura (nuvens)
                cloud_fraction = np.sum(lst_band < temp_threshold) / lst_band.size

                if cloud_fraction >= min_cloud_fraction:
                    cloudy_stats[city_name]["cloudy"] += 1
            except Exception as e:
                # Caso ocorra algum erro na leitura do arquivo específico
                continue

    # Ordena o ranking decrescente pelo número de fotos nubladas
    ranking = sorted(
        cloudy_stats.items(),
        key=lambda item: item[1]["cloudy"] / item[1]["total"],
        reverse=True
    )

    # Exibe o resultado formatado em tabela
    print("\n" + "="*65)
    print(f"{'RANKING DE CIDADES COM MAIS FOTOS NUBLADAS':^65}")
    print("="*65)
    print(f"{'Pos.':<6} | {'Cidade':<25} | {'Nubladas':<10} | {'Total':<10} | {'% Nublado':<10}")
    print("-" * 65)

    for idx, (city, stats) in enumerate(ranking, start=1):
        total = stats["total"]
        cloudy = stats["cloudy"]
        pct = (cloudy / total * 100) if total > 0 else 0
        print(f"{idx:<6} | {city:<25} | {cloudy:<10} | {total:<10} | {pct:.1f}%")

    print("="*65)
    return ranking

# ============================================================
# COMO EXECUTAR NO SEU SCRIPT:
# ============================================================
if __name__ == "__main__":
    # Certifique-se de que a variável DATA_DIR está definida
    ranking_cidades = analyze_cloudy_cities("/content/drive/MyDrive/heat_dataset_v3", temp_threshold=15.0, min_cloud_pct=30.0)

In [ ]:
import gc
import torch

# 1. Remova a referência ao objeto
del model
# 2. Forçe o Python a liberar a memória (o PyTorch detecta isso)
gc.collect()

# 3. (Opcional) Limpe o cache residual na GPU
torch.cuda.empty_cache()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

def analisar_alucinacao(model_path, fold_idx, data_root, all_cities, valid_files_set, global_stats):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Extrai média e desvio padrão reais da Banda LST (índice 4)
    mean_lst = global_stats[0][4].item()
    std_lst = global_stats[1][4].item()

    # 1. Recuperar o split exato do fold usando a mesma semente do treino (SEED=42)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(kf.split(all_cities))
    _, val_idx = splits[fold_idx]

    val_cities = [all_cities[i] for i in val_idx]

    # 2. Criação do Dataset e Loader de Validação
    val_ds = UrbanHeatDataset(
        data_root,
        allowed_cities=val_cities,
        valid_files_set=valid_files_set,
        stats=global_stats,
        is_train=False
    )
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=True)

    # 3. Carregar o Modelo Híbrido (agora com 4 canais fixos)
    model = HybridUrbanHeatNet(in_channels=4).to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    print(f"\n✅ Analisando modelo do Fold {fold_idx} ({model_path})...")

    # 4. Teste de Estabilidade (Alucinação via Ruído)
    seq, target, valid = next(iter(val_loader))
    seq, valid = seq.to(DEVICE), valid.to(DEVICE)

    with torch.no_grad():
        pred_base = model(seq)

        # Injeta ruído leve (1%) nos canais de entrada
        noise = torch.randn_like(seq) * 0.01
        pred_noisy = model(seq + noise)

        # Calcula a diferença absoluta ignorando o padding (valid_mask)
        valid_count = valid.sum() + 1e-8
        diff_abs_norm = (torch.abs(pred_base - pred_noisy) * valid).sum() / valid_count

        # Multiplica pelo desvio padrão global para obter o erro real em °C
        diff_celsius = diff_abs_norm.item() * std_lst

    print(f"📊 Estabilidade: Variação média de {diff_celsius:.6f} °C sob ruído leve")

    # 5. Plotagem da Física Aprendida
    plot_physical_correlation(model, val_loader, DEVICE, mean_lst, std_lst)

def plot_physical_correlation(model, loader, device, mean_lst, std_lst):
    all_ndvi, all_preds = [], []

    with torch.no_grad():
        for seq, target, valid in loader:
            seq, valid = seq.to(device), valid.to(device)
            pred = model(seq)

            # Extrai o NDVI (Canal 0 na construção do out_seq)
            ndvi = seq[:, 0, :, :].squeeze()

            # Desfaz a normalização da predição para graus Celsius reais!
            pred_celsius = (pred.squeeze() * std_lst) + mean_lst
            valid_mask = valid.squeeze().bool()

            # Pega APENAS os pixels válidos, ignorando os zeros do padding
            ndvi_valid = ndvi[valid_mask].cpu().numpy()
            pred_valid = pred_celsius[valid_mask].cpu().numpy()

            # Sub-amostragem: Pega 1 pixel a cada 50 para evitar sobrecarga de memória
            all_ndvi.extend(ndvi_valid[::50])
            all_preds.extend(pred_valid[::50])

            # Limita a 10.000 pontos totais para o gráfico renderizar rápido
            if len(all_ndvi) > 10000:
                break

    plt.figure(figsize=(8, 5))
    plt.scatter(all_ndvi, all_preds, alpha=0.15, s=4, color='red')
    plt.xlabel("NDVI (Padronizado via Z-score)")
    plt.ylabel("Temperatura Prevista (°C)")
    plt.title("Relação Física: NDVI vs LST Prevista")

    # Adiciona linha de tendência
    z = np.polyfit(all_ndvi, all_preds, 1)
    p = np.poly1d(z)
    plt.plot(all_ndvi, p(all_ndvi), "k--", alpha=0.8, label=f"Tendência: y={z[0]:.2f}x + {z[1]:.2f}")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- COMO CHAMAR ---
# Cole isso no final do seu script principal, fora do laço do K-Fold
if __name__ == "__main__":
    # Garante que as variáveis já existam no ambiente de execução
    try:
        analisar_alucinacao(
            model_path="best_hybrid_fold_0.pt", # Nome do arquivo do modelo salvo
            fold_idx=0,
            data_root=DATA_DIR,
            all_cities=all_cities,
            valid_files_set=valid_files_set,
            global_stats=global_stats
        )
    except NameError:
        print("⚠️ Variáveis globais não encontradas. Execute o treino/carregamento primeiro.")

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import GroupKFold

# Certifique-se de importar suas classes e funções atualizadas aqui!
# from seu_script import AttentionUNetLST, RegressionGradCAM, UrbanHeatDataset, filter_cloudy_files_runtime, compute_min_max, plot_grad_cam, set_seed

all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]
if len(all_cities) < K_FOLDS: exit("Cidades insuficientes para K-Fold.")

# 1. Aplicando Filtro Global de Nuvens
all_files_raw = list(Path(DATA_DIR).rglob("*.npy"))
valid_files_set = filter_cloudy_files_runtime(all_files_raw, temp_threshold=15.0, min_cloud_pct=30.0)

if not valid_files_set: exit("Nenhum arquivo válido após o filtro de nuvens!")

print("1. Extraindo Estatísticas Globais (Apenas Arquivos Válidos)...")
base_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=all_cities, valid_files_set=valid_files_set, normalize=True, is_train=False)
global_stats = (base_ds.mean, base_ds.std)

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

print("\n2. Iniciando K-Fold Cross Validation Espacial...")
for fold, (train_idx, val_idx) in enumerate(kf.split(all_cities)):
    train_cities = [all_cities[i] for i in train_idx]
    val_cities = [all_cities[i] for i in val_idx]

    train_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=train_cities, valid_files_set=valid_files_set, stats=global_stats, is_train=True)
    val_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=val_cities, valid_files_set=valid_files_set, stats=global_stats, is_train=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = HybridUrbanHeatNet().to(DEVICE)

    # GERAR GRAD-CAM DO MELHOR MODELO NO FOLD (Com Correção de Dimensões)
    model.load_state_dict(torch.load(f"best_hybrid_fold_{fold}.pt", weights_only=True))
    cam_generator = RegressionGradCAM(model, target_layer=model.up1.conv.conv2)
    sample_seq, sample_tgt, _ = next(iter(val_loader))
    # CORREÇÃO APLICADA: Pega o antepenúltimo e o penúltimo do lote, mantendo os 5 canais
    cam_map, pred_np = cam_generator.generate(sample_seq[0:1, :5, :, :].to(DEVICE))
    plot_grad_cam(sample_seq[0:1], cam_map, pred_np, sample_tgt[0, 0].numpy(), save_path=f"gradcam_fold_{fold}.png")
    print(f"-> Imagem do Grad-CAM salva como gradcam_fold_{fold}.png")

    break # Testando apenas o fold 1. Remova o break para rodar todos.

In [ ]:
import torch
import numpy as np
from pathlib import Path

# ============================================================
# CONFIGURAÇÕES
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "best_model_fold_0.pt"     # altere para o fold desejado
DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

# As mesmas cidades usadas como validação no fold
VAL_CITIES = [
    # coloque aqui as cidades daquele fold
    "Teresina",
    "Parnaiba",
    "Pedro_II",
    "Juazeiro_do_Norte_CE",
    "Imperatriz_MA"
]

# ============================================================
# IMPORTA O MODELO
# ============================================================

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

model = UrbanHeatNet().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Modelo carregado!")

# ============================================================
# CALCULA O DESVIO PADRÃO DAS ANOMALIAS
# ============================================================

stds = []

for city in VAL_CITIES:

    folder = Path(DATA_DIR) / city

    files = sorted(folder.glob("*.npy"))

    for file in files:

        img = np.load(file).astype(np.float32)

        img = ensure_channels_first(img)[CHANNELS_TO_KEEP]

        lst = img[CH_LST]

        lst = np.clip(lst, LST_CLIP_MIN, LST_CLIP_MAX)

        macro = lst.mean()

        anomaly = lst - macro

        std = anomaly.std()

        stds.append(std)

stds = np.array(stds)

print(f"Número de imagens : {len(stds)}")
print(f"Desvio médio       : {stds.mean():.4f} °C")
print(f"Desvio mediano     : {np.median(stds):.4f} °C")
print(f"Desvio mínimo      : {stds.min():.4f} °C")
print(f"Desvio máximo      : {stds.max():.4f} °C")

# ============================================================
# CONVERSÃO
# ============================================================

MAE_NORMALIZADO = 0.44

mae_real = MAE_NORMALIZADO * stds.mean()

print()
print("="*40)
print(f"MAE normalizado : {MAE_NORMALIZADO:.4f}")
print(f"MAE em LST      : {mae_real:.4f} °C")
print("="*40)

In [ ]:
import os

import random

import numpy as np

from pathlib import Path

from sklearn.model_selection import KFold

import math



import torch

import torch.nn as nn

import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader



# ============================================================

# CONFIGURAÇÃO GERAL

# ============================================================

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)



DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"



INPUT_MONTHS = 6

TOTAL_RAW_CHANNELS = 5  # LST, NDVI, NDBI, NDWI, DEM no .npy bruto



# Mantemos todos os 5 canais (água é carregada, mas escondida da rede)

CHANNELS_TO_KEEP = [0, 1, 2, 3, 4]

INPUT_CHANNELS = len(CHANNELS_TO_KEEP)



CH_NDVI = 0

CH_NDWI = 1

CH_NDBI = 2

CH_DEM = 3

CH_LST = 4



BATCH_SIZE = 2

EPOCHS = 20

LR = 1e-4

WEIGHT_DECAY = 0.05

BASE_CHANNELS = 64

K_FOLDS = 5



# Limites físicos

LST_CLIP_MIN, LST_CLIP_MAX = -20.0, 60.0





# ============================================================

# UTILITÁRIO DE FORMATO

# ============================================================

def ensure_channels_first(img):

    if img.ndim != 3:

        raise ValueError(f"Esperado array 3D, recebido shape {img.shape}")

    if img.shape[0] == TOTAL_RAW_CHANNELS:

        return img

    elif img.shape[-1] == TOTAL_RAW_CHANNELS:

        return np.transpose(img, (2, 0, 1))

    else:

        raise ValueError(

            f"Não foi possível identificar o eixo de canais no shape {img.shape}. "

            f"Esperava um eixo de tamanho {TOTAL_RAW_CHANNELS}."

        )





# ============================================================

# ESTATÍSTICAS

# ============================================================

def compute_dataset_statistics(files):

    count = 0

    sum_ = np.zeros(INPUT_CHANNELS, dtype=np.float64)

    sum_sq = np.zeros(INPUT_CHANNELS, dtype=np.float64)



    for f in files:

        img = np.load(f).astype(np.float32)

        img = ensure_channels_first(img)[CHANNELS_TO_KEEP, :, :]



        # Computa sobre a imagem inteira válida

        img[CH_LST] = np.clip(img[CH_LST], LST_CLIP_MIN, LST_CLIP_MAX)

        img[CH_NDVI] = np.clip(img[CH_NDVI], -1.0, 1.0)

        img[CH_NDWI] = np.clip(img[CH_NDWI], -1.0, 1.0)



        flat = img.reshape(INPUT_CHANNELS, -1).astype(np.float64)

        sum_ += flat.sum(axis=1)

        sum_sq += (flat ** 2).sum(axis=1)

        count += flat.shape[1]



    mean = sum_ / count

    var = np.clip(sum_sq / count - mean ** 2, a_min=0.0, a_max=None)

    std = np.sqrt(var)



    return mean.astype(np.float32), std.astype(np.float32)





def compute_delta_statistics(root, allowed_cities):

    root = Path(root)

    deltas = []



    for city in allowed_cities:

        folder = root / city

        if not folder.exists(): continue

        city_files = sorted(folder.glob("*.npy"))

        if len(city_files) < 2: continue



        for i in range(len(city_files) - 1):

            img_a = ensure_channels_first(np.load(city_files[i]).astype(np.float32))[CH_LST]

            img_b = ensure_channels_first(np.load(city_files[i + 1]).astype(np.float32))[CH_LST]



            h = min(img_a.shape[0], img_b.shape[0])

            w = min(img_a.shape[1], img_b.shape[1])

            img_a = np.clip(img_a[:h, :w], LST_CLIP_MIN, LST_CLIP_MAX)

            img_b = np.clip(img_b[:h, :w], LST_CLIP_MIN, LST_CLIP_MAX)



            deltas.append((img_b - img_a).flatten())



    if not deltas:

        raise ValueError("Nenhum par consecutivo encontrado.")



    deltas = np.concatenate(deltas)

    return float(np.mean(deltas)), float(np.std(deltas))





class Normalize:

    def __init__(self, mean, std):

        self.mean = torch.tensor(mean, dtype=torch.float32).view(INPUT_CHANNELS, 1, 1)

        self.std = torch.tensor(std, dtype=torch.float32).view(INPUT_CHANNELS, 1, 1) + 1e-6



    def __call__(self, img):

        if isinstance(img, np.ndarray):

            img = torch.from_numpy(img).float()

        return (img - self.mean) / self.std





# ============================================================

# DATASET COM RANDOM CROPPING E AUGMENTATION

# ============================================================

class UrbanHeatDataset(Dataset):

    def __init__(self, root, allowed_cities, normalize=True, stats=None, delta_stats=None, is_train=False):

        self.root = Path(root)

        self.normalize = normalize

        self.is_train = is_train

        self.samples = []

        all_files = []



        for city in allowed_cities:

            folder = self.root / city

            if not folder.exists(): continue

            files = sorted(folder.glob("*.npy"))

            all_files.extend(files)

            # Precisamos de apenas 2 imagens: a atual (geometria) e a futura (alvo)

            if len(files) < 2: continue

            for i in range(len(files) - 1):

                self.samples.append((files[i], files[i + 1]))



        if normalize and len(all_files) > 0:

            if stats is not None:

                self.mean, self.std = stats

            else:

                stats_file = self.root / "dataset_stats_all_channels.npz"

                if stats_file.exists():

                    saved = np.load(stats_file)

                    self.mean, self.std = saved["mean"], saved["std"]

                else:

                    self.mean, self.std = compute_dataset_statistics(all_files)

                    np.savez(stats_file, mean=self.mean, std=self.std)

            self.normalizer = Normalize(self.mean, self.std)



    def __len__(self):

        return len(self.samples)



    def __getitem__(self, idx):

        base_file, target_file = self.samples[idx]



        # 1. Carrega as imagens

        base_img = np.load(base_file).astype(np.float32)

        base_img = ensure_channels_first(base_img)[CHANNELS_TO_KEEP]

        tgt_img = np.load(target_file).astype(np.float32)

        tgt_img = ensure_channels_first(tgt_img)[CHANNELS_TO_KEEP]



        max_h = min(base_img.shape[1], tgt_img.shape[1])

        max_w = min(base_img.shape[2], tgt_img.shape[2])



        # 2. Crop

        crop_size = 400

        if self.is_train and max_h > crop_size and max_w > crop_size:

            y = random.randint(0, max_h - crop_size)

            x = random.randint(0, max_w - crop_size)

        else:

            y = (max_h - crop_size) // 2 if max_h > crop_size else 0

            x = (max_w - crop_size) // 2 if max_w > crop_size else 0



        # Aplica crop e normalização na base

        cropped_base = base_img[:, y:y+crop_size, x:x+crop_size]

        padded_base = np.zeros((INPUT_CHANNELS, crop_size, crop_size), dtype=np.float32)

        padded_base[:, :cropped_base.shape[1], :cropped_base.shape[2]] = cropped_base



        padded_base[CH_LST] = np.clip(padded_base[CH_LST], LST_CLIP_MIN, LST_CLIP_MAX)

        padded_base[CH_NDVI] = np.clip(padded_base[CH_NDVI], -1.0, 1.0)

        padded_base[CH_NDWI] = np.clip(padded_base[CH_NDWI], -1.0, 1.0)



        if self.normalize:

            padded_base = self.normalizer(padded_base)



        if isinstance(padded_base, np.ndarray):

            padded_base = torch.from_numpy(padded_base).float()



        # 3. EXTRAÇÃO DE FEATURES (O NDWI vai para o final do batch como máscara)

        model_features = padded_base[[CH_NDVI, CH_NDBI, CH_DEM], :, :] # 3 canais pra rede

        water_mask = padded_base[[CH_NDWI], :, :] # 1 canal pro Algoritmo Genético



        # 4. Sazonalidade

        try:

            nome_arquivo = target_file.stem

            mes_str = "".join(filter(str.isdigit, nome_arquivo.split('_')[-1]))

            mes_alvo = int(mes_str) if mes_str else (idx % 12) + 1

            if not (1 <= mes_alvo <= 12): mes_alvo = (idx % 12) + 1

        except Exception:

            mes_alvo = (idx % 12) + 1



        seno_mes = math.sin(2.0 * math.pi * mes_alvo / 12.0)

        cos_mes = math.cos(2.0 * math.pi * mes_alvo / 12.0)



        mapa_seno = torch.full((1, crop_size, crop_size), seno_mes, dtype=torch.float32)

        mapa_cos = torch.full((1, crop_size, crop_size), cos_mes, dtype=torch.float32)



        # Concatena: [NDVI(0), NDBI(1), DEM(2), Seno(3), Cosseno(4), NDWI(5)]

        out_seq = torch.cat([model_features, mapa_seno, mapa_cos, water_mask], dim=0)



        # 5. O ALVO: ANOMALIA ESPACIAL (Microclima)

        cropped_tgt = tgt_img[CH_LST, y:y+crop_size, x:x+crop_size]

        padded_tgt_lst = np.zeros((crop_size, crop_size), dtype=np.float32)

        padded_tgt_lst[:cropped_tgt.shape[0], :cropped_tgt.shape[1]] = cropped_tgt

        padded_tgt_lst = np.clip(padded_tgt_lst, LST_CLIP_MIN, LST_CLIP_MAX)



        tgt_tensor = torch.from_numpy(padded_tgt_lst)



        # Subtrai a média da cidade para isolar a Ilha de Calor

        macro_climate = tgt_tensor.mean()

        spatial_anomaly = tgt_tensor - macro_climate



        # Normaliza o desvio para a rede neural

        std_anomaly = spatial_anomaly.std() + 1e-6

        out_tgt = (spatial_anomaly / std_anomaly).unsqueeze(0).float()



        # Augmentation

        if self.is_train:

            if random.random() > 0.5:

                out_seq = torch.flip(out_seq, (-1,))

                out_tgt = torch.flip(out_tgt, (-1,))

            if random.random() > 0.5:

                out_seq = torch.flip(out_seq, (-2,))

                out_tgt = torch.flip(out_tgt, (-2,))

            k = random.randint(0, 3)

            if k > 0:

                out_seq = torch.rot90(out_seq, k, [-1, -2])

                out_tgt = torch.rot90(out_tgt, k, [-1, -2])



        return out_seq, out_tgt



# ============================================================

# MÉTRICAS E FUNÇÃO DE PERDA ESTÁVEL

# ============================================================

def compute_mae(pred, target):

    return torch.mean(torch.abs(pred.float() - target.float()))



def compute_r2(pred, target):

    p, t = pred.float(), target.float()

    target_mean = torch.mean(t)

    ss_tot = torch.sum((t - target_mean) ** 2)

    ss_res = torch.sum((t - p) ** 2)

    if ss_tot == 0: return torch.tensor(0.0, device=p.device)

    return 1 - (ss_res / ss_tot)



def compute_pearson(pred, target):

    p, t = pred.float(), target.float()

    pred_mean = torch.mean(p)

    target_mean = torch.mean(t)

    pred_centered = p - pred_mean

    target_centered = t - target_mean



    cov = torch.sum(pred_centered * target_centered)

    std_pred = torch.sqrt(torch.sum(pred_centered ** 2))

    std_target = torch.sqrt(torch.sum(target_centered ** 2))



    denominator = std_pred * std_target

    if denominator < 1e-6: return torch.tensor(0.0, device=p.device)

    return cov / denominator



def spatial_gradient_loss(pred, target):

    dx_pred = pred[:, :, :, 1:] - pred[:, :, :, :-1]

    dy_pred = pred[:, :, 1:, :] - pred[:, :, :-1, :]



    dx_tgt = target[:, :, :, 1:] - target[:, :, :, :-1]

    dy_tgt = target[:, :, 1:, :] - target[:, :, :-1, :]



    loss_x = torch.mean(torch.abs(dx_pred - dx_tgt))

    loss_y = torch.mean(torch.abs(dy_pred - dy_tgt))



    return loss_x + loss_y



class PhysicsInformedLoss(nn.Module):

    def __init__(self, phys_weight=0.01, struct_weight=0.2):

        super().__init__()

        self.data_loss_fn = nn.HuberLoss(reduction='mean', delta=1.0)

        self.phys_weight = phys_weight

        self.struct_weight = struct_weight



        self.laplacian_kernel = torch.tensor([

            [0.0, 1.0, 0.0],

            [1.0, -4.0, 1.0],

            [0.0, 1.0, 0.0]

        ]).view(1, 1, 3, 3)



    def laplacian(self, x):

        kernel = self.laplacian_kernel.to(x.device)

        return F.conv2d(x, kernel, padding=1)



    def forward(self, pred, target, ndvi_map):

        data_loss = self.data_loss_fn(pred, target)

        struct_loss = spatial_gradient_loss(pred, target)

        lap_pred = self.laplacian(pred)

        phys_loss = torch.mean(torch.abs(lap_pred) * torch.relu(ndvi_map))



        total_loss = data_loss + (self.struct_weight * struct_loss) + (self.phys_weight * phys_loss)

        return total_loss, data_loss, phys_loss



class ConvBNAct(nn.Module):

    def __init__(self, in_ch, out_ch, stride=1, padding=1, dilation=1, kernel_size=3):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, dilation=dilation, bias=False),

            nn.GroupNorm(num_groups=8, num_channels=out_ch),

            nn.GELU()

        )

    def forward(self, x):

        return self.block(x)



class ResidualBlock(nn.Module):

    def __init__(self, ch):

        super().__init__()

        self.conv1 = ConvBNAct(ch, ch)

        self.drop = nn.Dropout2d(0.1)

        self.conv2 = nn.Sequential(

            nn.Conv2d(ch, ch, 3, padding=1, bias=False),

            nn.GroupNorm(num_groups=8, num_channels=ch)

        )

        self.act = nn.GELU()

    def forward(self, x):

        return self.act(self.conv2(self.drop(self.conv1(x))) + x)





# ============================================================

# ASPP (Atrous Spatial Pyramid Pooling)

# ============================================================

class ASPP(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.c1 = nn.Conv2d(in_ch, out_ch, 1, bias=False)

        self.c2 = ConvBNAct(in_ch, out_ch, padding=6, dilation=6)

        self.c3 = ConvBNAct(in_ch, out_ch, padding=12, dilation=12)

        self.c4 = ConvBNAct(in_ch, out_ch, padding=18, dilation=18)

        self.project = nn.Sequential(

            nn.Conv2d(out_ch * 4, out_ch, 1, bias=False),

            nn.GroupNorm(num_groups=8, num_channels=out_ch),

            nn.GELU(),

            nn.Dropout2d(0.2)

        )

    def forward(self, x):

        return self.project(torch.cat([self.c1(x), self.c2(x), self.c3(x), self.c4(x)], dim=1))



# ============================================================

# BLOCOS U-NET (ENCODER / DECODER)

# ============================================================

class DownBlock(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.block = nn.Sequential(

            ConvBNAct(in_ch, out_ch, stride=2),

            ResidualBlock(out_ch)

        )

    def forward(self, x):

        return self.block(x)



class UpBlock(nn.Module):

    def __init__(self, in_ch, skip_ch, out_ch):

        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

        self.conv = nn.Sequential(

            ConvBNAct(in_ch + skip_ch, out_ch),

            ResidualBlock(out_ch)

        )

    def forward(self, x, skip):

        x_up = self.up(x)

        return self.conv(torch.cat([x_up, skip], dim=1))



# ============================================================

# ARQUITETURA PRINCIPAL FUNDIDA

# ============================================================

class SpatialStem(nn.Module):

    def __init__(self, in_ch=5, out_ch=64): # Rede recebe 5 canais (NDWI omitido)

        super().__init__()

        self.net = nn.Sequential(

            ConvBNAct(in_ch, out_ch, kernel_size=3, padding=1),

            ResidualBlock(out_ch),

            ConvBNAct(out_ch, out_ch, kernel_size=3, padding=1)

        )



    def forward(self, x):

        return self.net(x)



class UrbanHeatNet(nn.Module):

    def __init__(self, base_channels=64):

        super().__init__()

        # Entrada tem 5 canais: [NDVI, NDBI, DEM, Seno, Cosseno]

        self.stem = SpatialStem(in_ch=5, out_ch=base_channels)



        self.down1 = DownBlock(base_channels, base_channels * 2)

        self.down2 = DownBlock(base_channels * 2, base_channels * 4)



        self.aspp = ASPP(base_channels * 4, base_channels * 4)



        self.up2 = UpBlock(base_channels * 4, base_channels * 2, base_channels * 2)

        self.up1 = UpBlock(base_channels * 2, base_channels, base_channels)



        self.head = nn.Sequential(

            nn.Conv2d(base_channels, base_channels // 2, kernel_size=3, padding=1),

            nn.GELU(),

            nn.Dropout2d(0.1),

            nn.Conv2d(base_channels // 2, 1, kernel_size=1, bias=True)

        )



    def forward(self, x):

        s1 = self.stem(x)

        s2 = self.down1(s1)

        deep = self.down2(s2)



        deep = self.aspp(deep)



        x_up = self.up2(deep, s2)

        x_up = self.up1(x_up, s1)



        return self.head(x_up)



# ============================================================

# ALGORITMO GENÉTICO COM MÁSCARA DE VIABILIDADE

# ============================================================

class GeneticPlantingSimulator:

    def __init__(self, model, device, dataset_stats, delta_stats=None):

        self.model = model

        self.device = device



        # Mapeamento com base nos índices do CHANNELS_TO_KEEP original

        self.mean_ndvi = dataset_stats[0][0]

        self.std_ndvi  = dataset_stats[1][0]



        self.mean_ndwi = dataset_stats[0][1]

        self.std_ndwi  = dataset_stats[1][1]



        self.mean_ndbi = dataset_stats[0][2]

        self.std_ndbi  = dataset_stats[1][2]



        # Define o kernel explicitamente como float32 para evitar conflitos no autocast

        self.spillover_kernel = torch.tensor([

            [0.15, 0.30, 0.15],

            [0.30, 1.00, 0.30],

            [0.15, 0.30, 0.15]

        ], dtype=torch.float32, device=device).view(1, 1, 3, 3)



        self.max_ndvi = 0.70  # Teto realista de NDVI urbano denso na região

        self.ndvi_impact = 0.35

        self.ndbi_reduction = 0.40



    def get_valid_planting_mask(self, sequence):

        recent_ndbi_z = sequence[0, 1, :, :].squeeze()

        recent_ndwi_z = sequence[0, 5, :, :].squeeze()



        # 1. Desnormaliza para valores físicos reais

        recent_ndbi_raw = (recent_ndbi_z * self.std_ndbi) + self.mean_ndbi

        recent_ndwi_raw = (recent_ndwi_z * self.std_ndwi) + self.mean_ndwi



        # 2. Aplica a restrição física do ARBORMAP

        valid_mask = (recent_ndwi_raw < 0.0) & (recent_ndbi_raw < 1.0)

        return torch.nonzero(valid_mask)



    def apply_intervention(self, sequence, coords_y, coords_x):

        seq_interv = sequence.clone()

        _, _, H, W = seq_interv.shape



        plant_mask = torch.zeros((1, 1, H, W), dtype=torch.float32, device=self.device)

        coords_y = coords_y.long()

        coords_x = coords_x.long()

        plant_mask[0, 0, coords_y, coords_x] = 1.0



        impact_map = F.conv2d(plant_mask, self.spillover_kernel, padding=1)

        impact_flat = impact_map.squeeze()



        current_ndvi_z = seq_interv[0, 0, :, :]



        # 1. Recupera o valor físico do NDVI

        current_ndvi_raw = (current_ndvi_z * self.std_ndvi) + self.mean_ndvi



        # 2. Calcula o crescimento no espaço físico

        room_to_grow = torch.clamp(self.max_ndvi - current_ndvi_raw, min=0.0)

        actual_boost_raw = torch.min(impact_flat * self.ndvi_impact, room_to_grow)



        # 3. Converte o impacto físico para a proporção Z-score

        boost_z_ndvi = actual_boost_raw / self.std_ndvi

        reduction_z_ndbi = (actual_boost_raw * self.ndbi_reduction) / self.std_ndbi



        # 4. Soma a proporção normalizada ao tensor

        seq_interv[0, 0, :, :] += boost_z_ndvi

        seq_interv[0, 1, :, :] -= reduction_z_ndbi



        return seq_interv



    @torch.no_grad()

    def evaluate_fitness(self, sequence, individuals):

        self.model.eval()

        device_type = 'cuda' if torch.cuda.is_available() else 'cpu'



        with torch.amp.autocast(device_type):

            base_delta = self.model(sequence[:, :5, :, :]).squeeze()



        fitness_scores = np.zeros(len(individuals), dtype=np.float32)



        for i, coords in enumerate(individuals):

            # --- CORREÇÃO AQUI ---

            # Garante que coords seja um tensor no device correto

            if not isinstance(coords, torch.Tensor):

                coords = torch.tensor(coords, device=self.device)



            y_coords, x_coords = coords[:, 0], coords[:, 1]

            seq_interv = self.apply_intervention(sequence, y_coords, x_coords)

            # ---------------------



            with torch.amp.autocast(device_type):

                interv_delta = self.model(seq_interv[:, :5, :, :]).squeeze()



            impact = (base_delta - interv_delta) * self.std_delta

            fitness_scores[i] = impact.sum().item()



        return fitness_scores



    def optimize_genetic(self, sequence, num_trees, pop_size=50, generations=100, seed=42):

        # 1. Garantia de Determinismo

        if seed is not None:

            torch.manual_seed(seed)

            if torch.cuda.is_available():

                torch.cuda.manual_seed_all(seed)

            np.random.seed(seed)

            random.seed(seed)



        # 2. Definição da máscara de viabilidade (Corrigindo o erro)

        H, W = sequence.shape[-2:]

        valid_indices = self.get_valid_planting_mask(sequence)

        num_valid = len(valid_indices)



        # Cria a máscara booleana para consulta O(1)

        valid_mask_bool = torch.zeros((H, W), dtype=torch.bool, device=self.device)

        valid_mask_bool[valid_indices[:, 0], valid_indices[:, 1]] = True



        # Restante da inicialização da população...

        population = []

        for _ in range(pop_size):

            if num_valid >= num_trees:

                idx = torch.randperm(num_valid, device=self.device)[:num_trees]

                pts = valid_indices[idx]

            else:

                H, W = sequence.shape[-2:]

                pts = torch.stack([

                    torch.randint(0, H, (num_trees,), device=self.device),

                    torch.randint(0, W, (num_trees,), device=self.device)

                ], dim=1)

            population.append(pts)



        best_solution, best_fitness = None, -float('inf')



        print(f"Evoluindo {pop_size} indivíduos por {generations} gerações (Seed: {seed})...")

        for gen in range(generations):

            scores = self.evaluate_fitness(sequence, population)



            max_idx = int(np.argmax(scores))

            if scores[max_idx] > best_fitness:

                best_fitness = scores[max_idx]

                best_solution = population[max_idx].clone()



            # OTIMIZAÇÃO: Elitismo e reconstrução da população

            new_population = [best_solution.clone()]



            while len(new_population) < pop_size:

                # 1. SELEÇÃO (Torneio para escolher 2 pais diferentes)

                i1, i2 = random.sample(range(pop_size), 2)

                parent1 = population[i1] if scores[i1] > scores[i2] else population[i2]



                i3, i4 = random.sample(range(pop_size), 2)

                parent2 = population[i3] if scores[i3] > scores[i4] else population[i4]



                # 2. CRUZAMENTO / CROSSOVER (Uniforme)

                # Mistura as árvores do pai 1 com as do pai 2 (~50% de chance para cada)

                child = parent1.clone()

                cross_mask = torch.rand(num_trees, device=self.device) > 0.5

                child[cross_mask] = parent2[cross_mask]



                # 3. MUTAÇÃO (Por gene/árvore)

                # Damos a CADA árvore ~5% de chance de mudar de lugar

                mutation_rate = 0.05

                for mut_idx in range(num_trees):

                    if random.random() < mutation_rate:

                        if random.random() < 0.5:

                            # Mutação Local (Explotação): Move de 1 a 3 pixels

                            dy, dx = random.randint(-3, 3), random.randint(-3, 3)

                            new_y = max(0, min(H-1, child[mut_idx, 0].item() + dy))

                            new_x = max(0, min(W-1, child[mut_idx, 1].item() + dx))



                            if valid_mask_bool[new_y, new_x]:

                                child[mut_idx] = torch.tensor([new_y, new_x], device=self.device)

                            else:

                                child[mut_idx] = valid_indices[random.randint(0, num_valid - 1)]

                        else:

                            # Mutação Global (Exploração): Salta para um lugar aleatório

                            child[mut_idx] = valid_indices[random.randint(0, num_valid - 1)]



                # 4. PREVENÇÃO DE COLISÃO (Impede empilhamento no mesmo pixel)

                # Converte coordenadas (y, x) em índices 1D para achar os valores únicos

                flat_coords = child[:, 0] * W + child[:, 1]

                unique_coords = torch.unique(flat_coords)



                if len(unique_coords) < num_trees:

                    # Descobre quantas árvores "sumiram" por sobreposição

                    num_missing = num_trees - len(unique_coords)



                    # Reconstrói a criança apenas com coordenadas únicas

                    new_child = torch.zeros_like(child)

                    new_child[:len(unique_coords), 0] = unique_coords // W

                    new_child[:len(unique_coords), 1] = unique_coords % W



                    # Sorteia novas posições aleatórias para preencher a cota

                    idx = torch.randperm(num_valid, device=self.device)[:num_missing]

                    new_child[len(unique_coords):] = valid_indices[idx]

                    child = new_child



                new_population.append(child)



            population = new_population # Agora população tem tamanho pop_size

            print(f"Gen {gen}: Best Fitness = {best_fitness:.4f}")



        return best_solution[:, 0].tolist(), best_solution[:, 1].tolist(), best_fitness



# ============================================================

# TREINAMENTO E VALIDAÇÃO

# ============================================================

def fit_fold(model, train_loader, val_loader, epochs, fold_idx, global_stats):

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)



    loss_fn = PhysicsInformedLoss(phys_weight=0.0)

    scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

    best_loss = float('inf')



    print(f"\n--- Iniciando FOLD {fold_idx + 1} ---")

    for epoch in range(epochs):

        model.train()

        train_loss = 0

        n_batches_t = 0



        for seq, target in train_loader:

            seq, target = seq.to(DEVICE), target.to(DEVICE)



            # O NDVI é o índice 0 da nova sequência

            ndvi_map = seq[:, 0, :, :].unsqueeze(1)



            # Cortamos a água (canal 5) e passamos apenas os 5 canais para a rede

            # print(f"Valor mínimo do NDWI na imagem: {seq[:, 5, :, :].min().item():.4f}")

            # print(f"Valor máximo do NDWI na imagem: {seq[:, 5, :, :].max().item():.4f}")

            model_input = seq[:, :5, :, :]



            optimizer.zero_grad()



            with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):

                pred = model(model_input)

                total_loss, _, _ = loss_fn(pred, target, ndvi_map)



            if not torch.isfinite(total_loss): continue



            scaler.scale(total_loss).backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)

            scaler.update()



            train_loss += total_loss.item()

            n_batches_t += 1



        train_loss /= max(n_batches_t, 1)



        # --- VALIDAÇÃO ---

        model.eval()

        v_loss, v_mae = 0, 0

        n_batches_v = 0



        all_preds = []

        all_targets = []



        for seq, target in val_loader:

            seq, target = seq.to(DEVICE), target.to(DEVICE)

            ndvi_map = seq[:, 0, :, :].unsqueeze(1)



            # Cortamos a água para validação também

            model_input = seq[:, :5, :, :]



            with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):

                pred = model(model_input)

                total_loss, _, _ = loss_fn(pred, target, ndvi_map)



            if not torch.isfinite(total_loss): continue



            v_loss += total_loss.item()

            v_mae += compute_mae(pred, target).item()

            n_batches_v += 1



            all_preds.append(pred.detach().cpu())

            all_targets.append(target.detach().cpu())



        n_v = max(n_batches_v, 1)

        v_loss /= n_v

        v_mae /= n_v



        all_preds_t = torch.cat(all_preds)

        all_targets_t = torch.cat(all_targets)



        v_r2 = compute_r2(all_preds_t, all_targets_t).item()

        v_pearson = compute_pearson(all_preds_t, all_targets_t).item()

        v_std_pred = torch.std(all_preds_t).item()

        v_std_target = torch.std(all_targets_t).item()



        scheduler.step(v_loss)



        print(f"Fold {fold_idx+1} | Epoch {epoch+1:02d} | T.Loss: {train_loss:.4f} | V.Loss: {v_loss:.4f} | MAE: {v_mae:.4f}")

        print(f"       -> R2: {v_r2:.4f} | Pearson: {v_pearson:.4f} | Std(Pred): {v_std_pred:.4f} vs Std(Target): {v_std_target:.4f}")



        if v_loss < best_loss:

            best_loss = v_loss

            checkpoint = {

                'model_state_dict': model.state_dict(),

                'mean': global_stats[0],

                'std': global_stats[1],

                'input_channels': 5 # Atualizado para 5

            }

            torch.save(checkpoint, f"best_model_fold_{fold_idx}.pt")



    return best_loss



def test_single_batch_overfit(model, train_loader, device, epochs=150):

    print("\n" + "="*60)

    print("🚀 INICIANDO TESTE DE OVERFIT EM UM ÚNICO BATCH")

    print("="*60)



    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    loss_fn = torch.nn.MSELoss()



    seq_batch, target_batch = next(iter(train_loader))

    seq_batch = seq_batch.to(device)

    target_batch = target_batch.to(device)



    target_std = torch.std(target_batch).item()

    print(f"Tamanho do Batch de Entrada: {seq_batch.shape}")

    print(f"Desvio Padrão do Alvo: {target_std:.4f}\n")



    model.eval()



    for epoch in range(epochs):

        optimizer.zero_grad()



        # O NDVI é o canal 0

        ndvi_map = seq_batch[:, 0, :, :].unsqueeze(1)



        # Corta o NDWI

        model_input = seq_batch[:, :5, :, :]



        with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):

            pred = model(model_input)

            total_loss = loss_fn(pred, target_batch)



        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()



        if epoch % 10 == 0 or epoch == epochs - 1:

            with torch.no_grad():

                mae = compute_mae(pred, target_batch).item()

                r2 = compute_r2(pred, target_batch).item()

                pearson = compute_pearson(pred, target_batch).item()

                pred_std = torch.std(pred).item()



            print(f"Epoch {epoch:03d} | Loss: {total_loss.item():.4f} | MAE: {mae:.4f} | R2: {r2:.4f} | Pearson: {pearson:.4f} | Std(Pred): {pred_std:.4f}")



    print("="*60 + "\n")



# ============================================================

# EXECUÇÃO PRINCIPAL

# ============================================================

if __name__ == "__main__":

    all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]



    if not all_cities:

        print("Nenhuma cidade encontrada no diretório!")

        exit()



    print("\n1. Extraindo Estatísticas Globais...")



    if len(all_cities) < K_FOLDS:

        print(f"Poucas cidades. O K-Fold exige pelo menos {K_FOLDS}.")

        exit()



    base_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=all_cities, normalize=True, is_train=False)

    global_stats = (base_ds.mean, base_ds.std)



    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    fold_results = []



    print("\n2. Iniciando K-Fold Cross Validation Espacial...")

    # all_cities_array = np.array(all_cities) # Converte para numpy array para facilitar o indexamento



    # for fold_idx, (train_idx, val_idx) in enumerate(kf.split(all_cities_array)):

    #     # Seleciona os nomes usando os índices retornados pelo KFold

    #     train_cities = all_cities_array[train_idx]

    #     val_cities = all_cities_array[val_idx]



    #     print(f"\nFold {fold_idx + 1}:")

    #     print(f"  Treino ({len(train_cities)} cidades): {', '.join(train_cities)}")

    #     print(f"  Validação ({len(val_cities)} cidades): {', '.join(val_cities)}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_cities)):

        # Remoção do break "if fold < 4", agora treina todos.

        # if fold < 1:

        #   continue



        train_cities = [all_cities[i] for i in train_idx]

        val_cities = [all_cities[i] for i in val_idx]



        train_files = []

        for city in train_cities:

            folder = Path(DATA_DIR) / city

            if folder.exists():

                train_files.extend(sorted(folder.glob("*.npy")))



        print(f"Calculando estatísticas de normalização ({len(train_files)} arquivos de treino)...")

        fold_mean, fold_std = compute_dataset_statistics(train_files)



        fold_stats = (fold_mean, fold_std)



        train_ds = UrbanHeatDataset(

            DATA_DIR, allowed_cities=train_cities,

            stats=fold_stats, is_train=True

        )

        val_ds = UrbanHeatDataset(

            DATA_DIR, allowed_cities=val_cities,

            stats=fold_stats, is_train=False

        )



        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = UrbanHeatNet().to(DEVICE)



        for m in model.modules():

            if isinstance(m, (nn.Conv2d, nn.Conv3d)):

                torch.nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='leaky_relu')

                if m.bias is not None: torch.nn.init.zeros_(m.bias)



        best_val = fit_fold(model, train_loader, val_loader, EPOCHS, fold, global_stats)

        fold_results.append(best_val)



        if fold == K_FOLDS - 1:

            print("\n3. Executando Algoritmo Genético no Fold de Teste...")

            model.load_state_dict(torch.load(f"best_model_fold_{fold}.pt", weights_only=False)['model_state_dict'])

            seq, _ = next(iter(val_loader))

            seq = seq[0:1].to(DEVICE)



            simulator = GeneticPlantingSimulator(model, DEVICE, global_stats)

            opt_y, opt_x, opt_cool_sum = simulator.optimize_genetic(seq, num_trees=50, pop_size=50, generations=100)



            with torch.no_grad():

                base_delta = simulator.model(seq[:, :5, :, :]).squeeze()

                seq_interv = simulator.apply_intervention(seq, opt_y, opt_x)

                interv_delta = simulator.model(seq_interv[:, :5, :, :]).squeeze()



                mapa_resfriamento = (base_delta - interv_delta) * simulator.std_delta

                resfriamento_total_celsius = mapa_resfriamento[mapa_resfriamento > 0.01].sum().item()



                print(f"\n--- Otimização Final  com Restrição Geográfica ---")

                print(f"Resfriamento Total do Projeto: {resfriamento_total_celsius:.2f} °C acumulados")

                print(f"Eficiência Média por Célula: {resfriamento_total_celsius/50:.3f} °C")



    print(f"\nValidação Cruzada Concluída. Média Val Loss: {np.mean(fold_results):.4f} ± {np.std(fold_results):.4f}")

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 0. CONFIGURAÇÃO INICIAL E CARREGAMENTO
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

# Carregamento do Modelo
checkpoint = torch.load("best_model_fold_0.pt", map_location=device, weights_only=False)

in_channels = checkpoint.get('input_channels', 5)
model = UrbanHeatNet(base_channels=64).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Extração de Estatísticas
CH_LST = 4
stats_mean = checkpoint['mean']
stats_std = checkpoint['std']

MACRO_MEAN_FOLD_1 = stats_mean[CH_LST].item() if torch.is_tensor(stats_mean) else stats_mean[CH_LST]
STD_TRAIN_FOLD_1 = stats_std[CH_LST].item() if torch.is_tensor(stats_std) else stats_std[CH_LST]

print(f"Modelo carregado! Canais: {in_channels} | LST Mean: {MACRO_MEAN_FOLD_1:.2f} | LST Std: {STD_TRAIN_FOLD_1:.4f}")

# ==========================================
# 1. AS TRÊS ESTRATÉGIAS DE ARBORIZAÇÃO
# ==========================================

def baseline_aleatorio(valid_indices, N_celulas):
    """Nível 0: Plantio randômico (Controle)."""
    if len(valid_indices) == 0:
        return []
    indices = np.random.choice(len(valid_indices), size=min(N_celulas, len(valid_indices)), replace=False)
    return valid_indices[indices].tolist()

def baseline_guloso_estatico(valid_indices, pred_anomalia_base, N_celulas):
    """Nível 1: LST Bruta. Escolhe os N pixels mais quentes ignorando sobreposição."""
    valid_mask = np.zeros_like(pred_anomalia_base)
    for y, x in valid_indices:
        valid_mask[y, x] = 1

    masked_heatmap = np.copy(pred_anomalia_base)
    masked_heatmap[valid_mask == 0] = -np.inf

    flat_indices = np.argsort(masked_heatmap.ravel())[-N_celulas:]
    y_coords, x_coords = np.unravel_index(flat_indices, masked_heatmap.shape)

    return list(zip(y_coords.tolist(), x_coords.tolist()))

def baseline_guloso_dinamico(model, seq_tensor, simulador, N_celulas):
    """
    Nível 2: UrbanHeatNet com Consciência Espacial (O Diferencial do Artigo).
    Usa predição dinâmica da U-Net combinada com um kernel de área de influência.
    """
    print(f"Executando Guloso Dinâmico (UrbanHeatNet) para {N_celulas} passos...")
    model.eval()

    H, W = seq_tensor.shape[-2:]
    valid_indices = simulador.get_valid_planting_mask(seq_tensor)
    valid_mask = torch.zeros((H, W), dtype=torch.bool, device=simulador.device)
    valid_mask[valid_indices[:, 0], valid_indices[:, 1]] = True

    ocupado = torch.zeros((H, W), dtype=torch.bool, device=simulador.device)
    pontos_plantados = []
    seq_atual = seq_tensor.clone()

    # Previsão inicial (Estado 0) para ancoragem térmica
    with torch.no_grad():
        anomalia_base = model(seq_atual[:, :5, :, :]).squeeze()

    # Kernel de Suavização (Simula a área de influência de uma árvore: 3x3 ou 5x5)
    # Isso faz a rede buscar o "centro" do calor, não um pico isolado
    kernel_size = 3
    pad = kernel_size // 2
    avg_kernel = (torch.ones((1, 1, kernel_size, kernel_size), device=simulador.device) / (kernel_size**2))

    for step in range(N_celulas):
        with torch.no_grad():
            anomalia_atual = model(seq_atual[:, :5, :, :]).squeeze()

        # 1. HÍBRIDO TÉRMICO: 30% foco no problema original, 70% foco na situação atual
        mapa_hibrido = (0.3 * anomalia_base) + (0.7 * anomalia_atual)

        # 2. CONSCIÊNCIA ESPACIAL: Suaviza o mapa para encontrar "bolsões" de calor
        mapa_unsqueeze = mapa_hibrido.unsqueeze(0).unsqueeze(0)
        mapa_suavizado = F.conv2d(mapa_unsqueeze, avg_kernel, padding=pad).squeeze()

        # 3. MÁSCARA: Remove locais inválidos e já ocupados
        busca_mask = valid_mask & (~ocupado)
        mapa_busca = mapa_suavizado.clone()
        mapa_busca[~busca_mask] = -float('inf')

        # 4. DECISÃO: Escolhe o centro da maior massa térmica viável
        flat_idx = torch.argmax(mapa_busca)
        best_y = (flat_idx // W).item()
        best_x = (flat_idx % W).item()

        pontos_plantados.append((best_y, best_x))
        # 4. DECISÃO: Escolhe o centro da maior massa térmica viável
        flat_idx = torch.argmax(mapa_busca)
        best_y = (flat_idx // W).item()
        best_x = (flat_idx % W).item()

        pontos_plantados.append((best_y, best_x))

        # --- NOVO: RAIO DE EXCLUSÃO (Ex: bloco de 5x5 pixels) ---
        raio_bloqueio = 2
        y_min = max(0, best_y - raio_bloqueio)
        y_max = min(H, best_y + raio_bloqueio + 1)
        x_min = max(0, best_x - raio_bloqueio)
        x_max = min(W, best_x + raio_bloqueio + 1)

        # Bloqueia a área vizinha para forçar o algoritmo a procurar o PRÓXIMO bolsão de calor
        ocupado[y_min:y_max, x_min:x_max] = True

        # 5. ATUALIZAÇÃO FÍSICA
        y_tensor = torch.tensor([best_y], device=simulador.device)
        x_tensor = torch.tensor([best_x], device=simulador.device)
        seq_atual = simulador.apply_intervention(seq_atual, y_tensor, x_tensor)

    return pontos_plantados

# ==========================================
# 2. AVALIAÇÃO E PLOTAGEM
# ==========================================

def avaliar_estrategia(model, seq_tensor, pontos_intervencao, simulador, anomalia_base):
    if not pontos_intervencao:
        return 0.0, 0.0, np.zeros_like(anomalia_base)

    y_coords = torch.tensor([p[0] for p in pontos_intervencao], device=simulador.device)
    x_coords = torch.tensor([p[1] for p in pontos_intervencao], device=simulador.device)

    seq_interv = simulador.apply_intervention(seq_tensor, y_coords, x_coords)

    with torch.no_grad():
        anomalia_interv = model(seq_interv[:, :5, :, :]).cpu().numpy()[0, 0]

    mapa_resfriamento = anomalia_base - anomalia_interv

    # --- NOVA MÉTRICA: RESFRIAMENTO ÚTIL ---
    # Define o limiar de Ilha de Calor (acima da média + 0.5 desvio padrão da cena)
    limiar_critico = np.mean(anomalia_base) + (np.std(anomalia_base) * 0.5)

    # Cria uma máscara booleana apontando apenas para as áreas críticas
    mascara_ilhas_calor = anomalia_base > limiar_critico

    # Soma o resfriamento APENAS nas áreas que eram ilhas de calor
    resfriamento_util = np.sum(mapa_resfriamento[mascara_ilhas_calor & (mapa_resfriamento > 0)])

    # A eficiência prévia antes da escala
    eficiencia_util = resfriamento_util / max(len(pontos_intervencao), 1)

    return resfriamento_util, eficiencia_util, mapa_resfriamento

def executar_analise_plantio(model, seq_tensor, simulador, std_train, N_celulas=50):
    with torch.no_grad():
        anomalia_base = model(seq_tensor[:, :5, :, :]).cpu().numpy()[0, 0]

    valid_indices = simulador.get_valid_planting_mask(seq_tensor)

    # Execução das 3 Estratégias
    pts_aleat = baseline_aleatorio(valid_indices, N_celulas)
    resf_aleat, ef_aleat, _ = avaliar_estrategia(model, seq_tensor, pts_aleat, simulador, anomalia_base)

    pts_estatico = baseline_guloso_estatico(valid_indices.cpu().numpy(), anomalia_base, N_celulas)
    resf_estatico, ef_estatico, _ = avaliar_estrategia(model, seq_tensor, pts_estatico, simulador, anomalia_base)

    pts_dinamico = baseline_guloso_dinamico(model, seq_tensor, simulador, N_celulas)
    resf_dinamico, ef_dinamico, mapa_resf_dinamico = avaliar_estrategia(model, seq_tensor, pts_dinamico, simulador, anomalia_base)

    # Escala para Graus Celsius baseada no desvio físico real do dataset de treino
    resf_aleat *= std_train
    resf_estatico *= std_train
    resf_dinamico *= std_train

    # A eficiência é calculada sobre o resfriamento escalado corrigido
    ef_aleat = resf_aleat / N_celulas
    ef_estatico = resf_estatico / N_celulas
    ef_dinamico = resf_dinamico / N_celulas

    # Impressão Acadêmica
    print("\n" + "="*85)
    print("TABELA 1: DESEMPENHO DAS ESTRATÉGIAS DE MITIGAÇÃO TÉRMICA (RESFRIAMENTO ÚTIL)")
    print("="*85)
    print(f"{'Estratégia (Política de Plantio)':<35} | {'Resfriamento Útil (°C)':<22} | {'Eficiência Útil (°C/célula)':<20}")
    print("-" * 85)
    print(f"{'1. Controle (Aleatório)':<35} | {resf_aleat:<22.2f} | {ef_aleat:<20.4f}")
    print(f"{'2. Guloso Estático (LST Bruta)':<35} | {resf_estatico:<22.2f} | {ef_estatico:<20.4f}")
    print(f"{'3. Guloso Dinâmico (UrbanHeatNet)':<35} | {resf_dinamico:<22.2f} | {ef_dinamico:<20.4f}")
    print("="*85)

    # --- Plotagem Espacial (O Gráfico do Artigo) ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    H, W = seq_tensor.shape[-2:]
    valid_mask_2d = np.zeros((H, W))
    for y, x in valid_indices:
        valid_mask_2d[y, x] = 1

    # FUNDO DE ALTO CONTRASTE
    # 1.0 (Branco puro) para áreas válidas
    # 0.15 (Cinza muito escuro/quase preto) para áreas inválidas (rios/construções densas)
    fundo = np.where(valid_mask_2d == 1, 1.0, 0.15)

    # ---------------------------------------------------------
    # Plot 1: Estratégia Estática
    # ---------------------------------------------------------
    axes[0].imshow(fundo, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title("Estratégia Estática: Foco Apenas na LST Bruta", pad=10)

    if pts_estatico:
        y_lst, x_lst = zip(*pts_estatico)
        # Marcador vermelho com linha mais grossa para destacar no fundo branco
        axes[0].scatter(x_lst, y_lst, c='red', marker='x', s=50, linewidths=2.5, label='Decisão (LST Bruta)')

    axes[0].legend(loc='upper right', framealpha=0.95)
    axes[0].axis('off') # Remove os eixos numéricos para visual mais limpo

    # ---------------------------------------------------------
    # Plot 2: Mapa Térmico e UrbanHeatNet
    # ---------------------------------------------------------
    axes[1].imshow(fundo, cmap='gray', vmin=0, vmax=1)

    # Prepara o mapa de resfriamento
    mapa_resfriamento_plot = np.where(mapa_resf_dinamico > 0, mapa_resf_dinamico * std_train, np.nan)
    # Garante que só será colorido onde a máscara é válida
    mapa_resf_visual = np.where(valid_mask_2d == 1, mapa_resfriamento_plot, np.nan)

    v_min = np.nanmin(mapa_resf_visual)
    v_max = np.nanmax(mapa_resf_visual)

    # Plota o resfriamento com alpha maior (0.85) para cores mais vivas
    im1 = axes[1].imshow(mapa_resf_visual, cmap='coolwarm_r', vmin=v_min, vmax=v_max, alpha=0.85)

    axes[1].set_title("Estratégia Dinâmica: UrbanHeatNet (Foco no Espalhamento)", pad=10)

    if pts_dinamico:
        y_unet, x_unet = zip(*pts_dinamico)
        # Marcador verde limão com borda preta forte para máximo destaque
        axes[1].scatter(x_unet, y_unet, c='#00FF00', marker='o', s=45, edgecolors='black', linewidth=1.2, label='Decisão (UrbanHeatNet)')

    axes[1].legend(loc='upper right', framealpha=0.95)
    axes[1].axis('off')

    # Ajuste da barra de cores
    cbar = fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    cbar.set_label("Resfriamento Projetado (°C)", rotation=270, labelpad=15)

    plt.tight_layout()
    plt.savefig("figura_estrategias_comparativas.png", dpi=300, bbox_inches='tight')
    plt.show()

# ==========================================
# EXECUÇÃO PRINCIPAL
# ==========================================
if __name__ == "__main__":
    from torch.utils.data import DataLoader

    CIDADE_ALVO = "Teresinar"
    BATCH_SIZE = 4

    cidade_path = Path(DATA_DIR) / CIDADE_ALVO
    if not cidade_path.exists():
        print(f"Erro: A pasta {cidade_path} não foi encontrada!")
        exit()

    fold_stats = (stats_mean, stats_std)

    val_ds = UrbanHeatDataset(
        root=DATA_DIR,
        allowed_cities=[CIDADE_ALVO],
        stats=fold_stats,
        is_train=False,
        normalize=True
    )

    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    if len(val_ds) > 0:
        seq_batch, _ = next(iter(val_loader))
        seq_tensor = seq_batch[0:1].to(device)

        simulador = GeneticPlantingSimulator(model, device, dataset_stats=fold_stats)

        print("\nIniciando Benchmark de Estratégias para o Artigo...")
        executar_analise_plantio(model, seq_tensor, simulador, STD_TRAIN_FOLD_1, N_celulas=50)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import geopandas as gpd
import requests
import io

# 1. Mapeamento e Numeração
folds_ibge = {
    # Fold 1: IDs 01 a 04 (Iguatu - CE, Parnaíba - PI, Pedro II - PI, Juazeiro do Norte - CE)
    1: ['2305506', '2207702', '2207900', '2307304'],

    # Fold 2: IDs 05 a 08 (Floriano - PI, Teresina - PI, São Raimundo Nonato - PI, Picos - PI)
    2: ['2203909', '2211001', '2210607', '2208007'],

    # Fold 3: IDs 09 a 12 (Barreiras - BA, Imperatriz - MA, Patos - PB, Crato - CE)
    3: ['2903201', '2105302', '2510808', '2304202'],

    # Fold 4: IDs 13 a 16 (Mossoró - RN, Bom Jesus - PI, Uruçuí - PI, Araguaína - TO)
    4: ['2402001', '2201903', '2211100', '1702109'],

    # Fold 5: IDs 17 a 20 (Sobral - CE, Corrente - PI, Vitória da Conquista - BA, Paulo Afonso - BA)
    5: ['2312908', '2202901', '2933307', '2924009']
}

# Criando lista plana de todos os códigos e o dicionário de números
all_codes = [cod for sublist in folds_ibge.values() for cod in sublist]
mapa_numero_cidade = {cod: i+1 for i, cod in enumerate(all_codes)}

# 2. Downloads (IBGE)
headers = {'User-Agent': 'Mozilla/5.0'}
url_estados = "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&intrarregiao=UF"
url_mun = "https://servicodados.ibge.gov.br/api/v3/malhas/paises/BR?formato=application/vnd.geo+json&intrarregiao=municipio"

br_estados = gpd.read_file(io.StringIO(requests.get(url_estados, headers=headers).text))
br_mun = gpd.read_file(io.StringIO(requests.get(url_mun, headers=headers).text))

# 3. Plotagem
fig, ax = plt.subplots(figsize=(12, 12))
br_estados.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.5)

colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00']

for fold_num, codigos in folds_ibge.items():
    mun_fold = br_mun[br_mun['codarea'].isin(codigos)]
    mun_fold.plot(ax=ax, color=colors[fold_num-1], edgecolor='black', linewidth=0.5, zorder=4)

    # Adicionando os números (labels)
    for _, row in mun_fold.iterrows():
        centroid = row.geometry.centroid
        ax.text(centroid.x, centroid.y, str(mapa_numero_cidade[row['codarea']]),
          fontsize=8, ha='center', va='center', color='white',
          fontweight='bold', zorder=10,
          path_effects=[pe.withStroke(linewidth=2, foreground='black')])

# Ajustes Finais
ax.set_xlim(-49, -34); ax.set_ylim(-16, -1)
legend_patches = [mpatches.Patch(color=colors[i-1], label=f'Fold {i}') for i in folds_ibge.keys()]
plt.legend(handles=legend_patches, title="Validação Cruzada", loc='lower right')
plt.title("Distribuição Espacial da Validação Cruzada (Índice Numérico)")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Dados extraídos do seu log de treinamento
epochs = list(range(1, 21))

t_loss = [0.3805, 0.3130, 0.2999, 0.2928, 0.2885, 0.2834, 0.2814, 0.2804, 0.2792, 0.2778,
          0.2757, 0.2756, 0.2761, 0.2769, 0.2756, 0.2761, 0.2735, 0.2725, 0.2702, 0.2694]

v_loss = [0.2944, 0.2721, 0.2685, 0.2641, 0.2587, 0.2591, 0.2563, 0.2635, 0.2556, 0.2551,
          0.2569, 0.2546, 0.2596, 0.2586, 0.2675, 0.2586, 0.2756, 0.2520, 0.2571, 0.2645]

r2 = [0.4406, 0.4845, 0.4830, 0.4924, 0.5049, 0.5066, 0.5062, 0.4930, 0.5085, 0.5056,
      0.5015, 0.5065, 0.4924, 0.5030, 0.4763, 0.5053, 0.4591, 0.5101, 0.4980, 0.4806]

pearson = [0.6744, 0.6964, 0.7015, 0.7042, 0.7131, 0.7145, 0.7121, 0.7074, 0.7134, 0.7138,
           0.7098, 0.7128, 0.7049, 0.7124, 0.6906, 0.7160, 0.6854, 0.7148, 0.7057, 0.6981]

std_pred = [0.7915, 0.7114, 0.7896, 0.7251, 0.6694, 0.6553, 0.6963, 0.6196, 0.6899, 0.7007,
            0.6812, 0.7188, 0.7700, 0.6440, 0.6636, 0.6313, 0.5811, 0.7422, 0.7095, 0.6260]

# O std do target é constante em todas as épocas
std_target = [0.9979] * 20

# Configurações globais de estilo
plt.style.use('seaborn-v0_8-whitegrid')

# ==============================================================================
# FIGURA 1: Curvas de treinamento (Loss, R² e Pearson)
# ==============================================================================
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# (a) Perda de Treino vs Validação
ax1.plot(epochs, t_loss, label='Treino (T.Loss)', marker='o', linestyle='-', color='tab:blue')
ax1.plot(epochs, v_loss, label='Validação (V.Loss)', marker='s', linestyle='-', color='tab:orange')
ax1.set_title('(a) Perda de Treino vs. Validação')
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss (MSE)')
ax1.set_xticks(epochs[::2]) # Mostrar ticks a cada 2 épocas
ax1.legend()

# (b) R² e Correlação de Pearson
ax2.plot(epochs, r2, label='R² (Validação)', marker='^', linestyle='-', color='tab:green')
ax2.plot(epochs, pearson, label='Pearson (Validação)', marker='d', linestyle='-', color='tab:purple')
ax2.set_title('(b) Métricas de Avaliação na Validação')
ax2.set_xlabel('Época')
ax2.set_ylabel('Valor da Métrica')
ax2.set_xticks(epochs[::2])
ax2.legend()

# fig1.suptitle('FIGURA 1 – Curvas de treinamento do Fold 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ==============================================================================
# FIGURA 2: Desvio-padrão da Predição vs Alvo
# ==============================================================================
fig2, ax3 = plt.subplots(figsize=(8, 5))

ax3.plot(epochs, std_pred, label='Std (Predição)', marker='o', linestyle='-', color='tab:red')
ax3.plot(epochs, std_target, label='Std (Alvo)', linestyle='--', linewidth=2.5, color='black')
# ax3.set_title('FIGURA 2 – Desvio-padrão: Predição vs. Alvo ao longo do treinamento', fontsize=12, fontweight='bold')
ax3.set_xlabel('Época')
ax3.set_ylabel('Desvio Padrão')
ax3.set_xticks(epochs[::2])
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

# ============================================================
# FUNÇÃO DE FEATURE ABLATION
# ============================================================
def perform_feature_ablation(model, val_loader, device):
    """
    Realiza a ablação de características zerando um canal por vez e
    medindo a degradação na performance (MAE e R2).
    """
    print("\n" + "="*60)
    print("🔬 INICIANDO FEATURE ABLATION (FOLD 0)")
    print("="*60)

    model.eval()

    # Nomes dos 6 canais na ordem exata em que entram no modelo
    feature_names = ['NDVI', 'NDBI', 'NDWI', 'DEM', 'Mes_Sin', 'Mes_Cos']
    num_features = len(feature_names)

    def evaluate_loader(loader, ablate_idx=None):
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for seq, target in loader:
                seq = seq.to(device)
                target = target.to(device)

                # Se ablate_idx for passado, zeramos o canal correspondente
                # Como os dados numéricos estão em Z-score, a média é 0.
                # Para seno e cosseno, a média anual também é 0.
                if ablate_idx is not None:
                    seq[:, ablate_idx, :, :] = 0.0

                with torch.amp.autocast('cuda' if torch.cuda.is_available() else 'cpu'):
                    pred = model(seq)

                all_preds.append(pred.detach().cpu())
                all_targets.append(target.detach().cpu())

        all_preds_t = torch.cat(all_preds)
        all_targets_t = torch.cat(all_targets)

        # Reaproveita as métricas já definidas no seu código
        mae = compute_mae(all_preds_t, all_targets_t).item()
        r2 = compute_r2(all_preds_t, all_targets_t).item()
        return mae, r2

    # 1. Avalia o Baseline (Todos os canais ativos)
    print("Calculando Baseline (Todos os features)...")
    baseline_mae, baseline_r2 = evaluate_loader(val_loader, ablate_idx=None)
    print(f"Baseline -> MAE: {baseline_mae:.4f} | R2: {baseline_r2:.4f}\n")

    # 2. Avalia a ablação para cada canal
    results = []
    for idx, feature in enumerate(feature_names):
        print(f"Ablacionando canal {idx}: {feature}...")
        mae, r2 = evaluate_loader(val_loader, ablate_idx=idx)

        # Calcula a degradação (quanto piorou o modelo sem essa feature)
        # Quedas maiores no R2 ou aumentos no MAE indicam maior importância
        r2_drop = baseline_r2 - r2
        mae_increase = mae - baseline_mae

        results.append({
            'feature': feature,
            'mae': mae,
            'r2': r2,
            'r2_drop': r2_drop,
            'mae_increase': mae_increase
        })

    # 3. Exibe o Ranking de Importância
    print("\n" + "-"*60)
    print("🏆 RANKING DE IMPORTÂNCIA DOS RECURSOS (Por Queda de R2)")
    print("-"*60)

    # Ordena os resultados pela maior queda no R2 (do mais importante para o menos)
    results_sorted = sorted(results, key=lambda x: x['r2_drop'], reverse=True)

    for rank, res in enumerate(results_sorted, 1):
        print(f"{rank}. {res['feature'].ljust(10)} | "
              f"R2 Após Ablação: {res['r2']:.4f} (Queda de {res['r2_drop']:.4f}) | "
              f"MAE Aumento: +{res['mae_increase']:.4f}")

# ============================================================
# SCRIPT DE CARREGAMENTO (EXECUÇÃO)
# ============================================================
if __name__ == "__main__":
    # Garantir a mesma seed para obter o mesmo Fold 0 do treinamento
    SEED = 42
    np.random.seed(SEED)

    all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]

    if len(all_cities) < K_FOLDS:
        print("Cidades insuficientes no diretório de dados.")
        exit()

    # Recriar as partições exatas do Fold 0
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    train_idx, val_idx = next(kf.split(all_cities)) # Pega apenas a primeira divisão (Fold 0)

    train_cities = [all_cities[i] for i in train_idx]
    val_cities = [all_cities[i] for i in val_idx]

    # Carregar o checkpoint do modelo do Fold 0
    model_path = "./best_model_fold_0.pt"
    if not Path(model_path).exists():
        print(f"Erro: Arquivo '{model_path}' não encontrado. Certifique-se de que o fold 0 concluiu o treinamento.")
        exit()

    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)

    # Extrair as estatísticas de normalização que foram salvas no checkpoint
    saved_stats = (checkpoint['mean'], checkpoint['std'])

    # Instanciar o Dataset de Validação com as cidades e stats corretos
    val_ds = UrbanHeatDataset(
        DATA_DIR,
        allowed_cities=val_cities,
        stats=saved_stats,
        is_train=False
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # Inicializar o Modelo e carregar os pesos
    model = UrbanHeatNet(base_channels=64).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])

    # Executar a análise
    perform_feature_ablation(model, val_loader, DEVICE)

In [ ]:
import os
stats_path = os.path.join(DATA_DIR, "dataset_stats_zscore_without_ndwi.npz")
if os.path.exists(stats_path):
    os.remove(stats_path)

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold

# Certifique-se de importar suas classes e variáveis do script principal!
# from seu_script_principal import UrbanHeatNet, UrbanHeatDataset, K_FOLDS, SEED

def get_fold_data(root, fold_idx, all_cities, global_stats):
    """Reconstrói apenas o dataset de validação necessário para o fold específico."""
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    splits = list(kf.split(all_cities))
    _, val_idx = splits[fold_idx]

    val_cities = [all_cities[i] for i in val_idx]
    # CORREÇÃO: Adicionado is_train=False para desligar o Augmentation na avaliação
    return UrbanHeatDataset(root, allowed_cities=val_cities, stats=global_stats, is_train=False)

def analisar_alucinacao(model_path, fold_idx, data_root):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)

    model = UrbanHeatNet().to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    all_cities = [d.name for d in Path(data_root).iterdir() if d.is_dir() and not d.name.startswith('.')]
    global_stats = (checkpoint['mean'], checkpoint['std'])

    # Extrai de forma segura o escalar da temperatura para estimar Celsius
    # O LST é o índice 0 na array original de estatísticas
    std_lst = float(checkpoint['std'][0].flatten()[0])

    val_ds = get_fold_data(data_root, fold_idx, all_cities, global_stats)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=True)

    print(f"✅ Analisando modelo do Fold {fold_idx}...")

    seq, _ = next(iter(val_loader))
    seq = seq.to(DEVICE)

    with torch.no_grad():
        # AQUI ESTÁ A MUDANÇA: Cortamos o canal 5 (NDWI) e passamos apenas 0 a 4
        # model_input = seq[:, :CH_NDWI, :, :]
        model_input = seq

    # with torch.no_grad():
        pred_base = model(model_input)

        # CORREÇÃO: Como o LST foi expulso da entrada, não há risco de vazar ruído
        # de temperatura na entrada. O tensor 'seq' tem 6 canais espaciais.
        noise = torch.randn_like(model_input) * 0.01
        pred_noisy = model(model_input + noise)

        diff_celsius = torch.abs(pred_base - pred_noisy).mean().item() * std_lst

    print(f"📊 Estabilidade: {diff_celsius:.6f} °C (variação média na predição sob ruído leve)")

    plot_physical_correlation(model, val_loader, DEVICE, std_lst)

def plot_physical_correlation(model, loader, device, std_lst):
    all_ndvi, all_preds = [], []

    with torch.no_grad():
        for seq, _ in loader:
            model_input = seq[:, :5, :, :]
            model_input = model_input.to(device)
            pred = model(model_input)

            # Desnormaliza a Anomalia prevista para escala aproximada (°C)
            pred_celsius = pred * std_lst

            # CORREÇÃO: A arquitetura 2D não tem dimensão de tempo.
            # O tensor 'seq' é (Batch, 6, H, W) onde NDVI agora é o canal 0.
            ndvi = seq[:, 0, :, :].cpu().numpy().flatten()
            pred_flat = pred_celsius.cpu().numpy().flatten()

            all_ndvi.extend(ndvi)
            all_preds.extend(pred_flat)
            if len(all_ndvi) > 5000:
                break

    plt.figure(figsize=(8, 5))
    plt.scatter(all_ndvi, all_preds, alpha=0.1, s=1, color='red')
    plt.xlabel("NDVI (Vegetação, normalizado Z-Score)")
    plt.ylabel("Anomalia de Temperatura Prevista (°C)")
    plt.title("Relação Física Universal: NDVI x Anomalia Térmica")
    plt.grid(True)
    plt.show()

# --- EXECUÇÃO ---
if __name__ == "__main__":
    analisar_alucinacao(
        model_path="best_model_fold_0.pt",
        fold_idx=0,
        data_root="/content/drive/MyDrive/heat_dataset_v3"
    )

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import random
from pathlib import Path
from torch.utils.data import DataLoader
import matplotlib.colors as mcolors

# ====================================================================
# 1. IMPORTAÇÕES DO SEU SCRIPT PRINCIPAL
# ====================================================================
# Substitua 'seu_script_treino' pelo nome do arquivo onde estão as classes.
# Se estiver no Colab no mesmo Notebook, não precisa importar, apenas rode a célula!
# try:
    # from seu_script_treino import UrbanHeatNet, UrbanHeatDataset, CH_LST, CH_NDVI, CH_NDBI, CH_NDWI
# except ImportError:
#     print("Aviso: Certifique-se de que UrbanHeatNet e UrbanHeatDataset estão na memória.")
#     # Índices da nova arquitetura 2D (apenas para garantir)
#     CH_NDVI, CH_NDBI, CH_NDWI = 0, 1, 2


# ====================================================================
# 2. SIMULADOR GENÉTICO (OTIMIZADO PARA ARQUITETURA 2D)
# ====================================================================
class GeneticPlantingSimulator:
    def __init__(self, model, device, std_lst):
        self.model = model
        self.device = device
        self.std_lst = std_lst  # Fator de conversão para °C

        # Kernel de espalhamento (árvores resfriam os vizinhos diretos)
        self.spillover_kernel = torch.tensor([
            [0.10, 0.25, 0.10],
            [0.25, 1.00, 0.25],
            [0.10, 0.25, 0.10]
        ], device=device).view(1, 1, 3, 3)

    def get_valid_planting_mask(self, seq):
        """Bloqueia o plantio na água (NDWI alto) e foca em áreas não-vegetadas (NDVI baixo)."""
        ndvi = seq[0, CH_NDVI, :, :]
        ndwi = seq[0, CH_NDWI, :, :]

        # O limiar depende da sua normalização Z-score.
        # ndwi < 0.0 geralmente significa solo seco/asfalto.
        valid_mask = ndvi < 0.5
        return torch.nonzero(valid_mask)

    def apply_intervention(self, seq, coords_y, coords_x):
        """Planta árvores virtuais e espalha o efeito de vegetação localmente."""
        seq_interv = seq.clone()
        _, _, H, W = seq_interv.shape
        plant_mask = torch.zeros((1, 1, H, W), device=self.device)

        for y, x in zip(coords_y, coords_x):
            if 0 <= y < H and 0 <= x < W:
                plant_mask[0, 0, y, x] = 1.0

        # Espalha o efeito da copa das árvores para pixels vizinhos
        impact_map = F.conv2d(plant_mask, self.spillover_kernel, padding=1).squeeze(0)

        # Aumenta vigorosamente o NDVI (Canal 0) e diminui Prédios/Asfalto (Canal 1)
        # Limitamos o NDVI máximo para não ultrapassar a realidade física
        current_ndvi = seq_interv[0, CH_NDVI, :, :]
        room_to_grow = torch.clamp(2.0 - current_ndvi, min=0.0)
        actual_boost = torch.min(impact_map[0] * 1.5, room_to_grow)

        seq_interv[0, CH_NDVI, :, :] += actual_boost
        seq_interv[0, CH_NDBI, :, :] -= actual_boost * 0.5

        return seq_interv

    @torch.no_grad()
    def evaluate_fitness(self, seq, population):
        self.model.eval()
        # Predição base antes de plantar qualquer árvore
        base_pred = self.model(seq).squeeze()

        scores = []
        for coords in population:
            y_coords, x_coords = coords[:, 0], coords[:, 1]

            # Predição com as novas árvores
            seq_interv = self.apply_intervention(seq, y_coords, x_coords)
            interv_pred = self.model(seq_interv).squeeze()

            # Resfriamento em Graus Celsius = (Base - Intervenção) * Desvio Padrão Real
            # Valores Positivos = Temperatura Caiu (Sucesso)
            cooling_map = (base_pred - interv_pred) * self.std_lst

            # O fitness é o somatório do resfriamento gerado (penaliza se esquentar)
            score = cooling_map[cooling_map > 0.01].sum().item()
            scores.append(score)

        return np.array(scores)

    def optimize(self, seq, num_trees=50, pop_size=20, generations=15):
        valid_indices = self.get_valid_planting_mask(seq)

        if len(valid_indices) < num_trees:
            print("⚠️ Poucos espaços válidos! Reduzindo número de árvores.")
            num_trees = len(valid_indices)

        # 1. População Inicial Aleatória
        population = []
        for _ in range(pop_size):
            idx = torch.randperm(len(valid_indices))[:num_trees]
            population.append(valid_indices[idx])

        best_solution = None
        best_fitness = -float('inf')

        print(f"🌱 Iniciando Otimização Genética ({generations} Gerações)...")
        for gen in range(generations):
            scores = self.evaluate_fitness(seq, population)
            max_idx = np.argmax(scores)

            if scores[max_idx] > best_fitness:
                best_fitness = scores[max_idx]
                best_solution = population[max_idx].clone()

            print(f"   ➜ Geração {gen+1:02d} | Resfriamento Máx Projetado: {best_fitness:.2f} °C Acumulados")

            # 2. Crossover e Mutação
            new_pop = []
            for _ in range(pop_size):
                # Torneio: Escolhe o melhor entre 2 indivíduos aleatórios
                i1, i2 = random.sample(range(pop_size), 2)
                parent = population[i1] if scores[i1] > scores[i2] else population[i2]
                child = parent.clone()

                # Mutação: Troca algumas árvores de lugar
                if random.random() < 0.6:
                    mut_count = random.randint(1, max(1, num_trees // 4))
                    mut_indices = random.sample(range(num_trees), mut_count)
                    new_spots = valid_indices[torch.randperm(len(valid_indices))[:mut_count]]
                    child[mut_indices] = new_spots

                new_pop.append(child)
            population = new_pop

        return best_solution, best_fitness


# ====================================================================
# 3. FUNÇÃO DE VISUALIZAÇÃO E EXECUÇÃO PRINCIPAL
# ====================================================================
def executar_simulacao(model_path, data_root, num_trees=100):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("\nCarregando Modelo...")

    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model = UrbanHeatNet().to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # O LST estava no índice 0 no array global de estatísticas
    std_lst = float(checkpoint['std'][0].flatten()[0])

    print("Carregando Dados de Validação...")
    all_cities = [d.name for d in Path(data_root).iterdir() if d.is_dir() and not d.name.startswith('.')]

    # Pega qualquer cidade de validação (ex: a primeira da lista)
    val_ds = UrbanHeatDataset(data_root, allowed_cities=[all_cities[0]],
                              stats=(checkpoint['mean'], checkpoint['std']), is_train=False)

    val_loader = DataLoader(val_ds, batch_size=1, shuffle=True)
    seq, target = next(iter(val_loader))
    seq = seq.to(DEVICE)

    # Inicia o Simulador
    simulator = GeneticPlantingSimulator(model, DEVICE, std_lst)

    # Roda a otimização
    best_coords, total_cooling = simulator.optimize(seq, num_trees=num_trees, pop_size=20, generations=20)

    # ================== RESULTADOS VISUAIS ==================
    with torch.no_grad():
        base_pred = model(seq).squeeze()
        seq_interv = simulator.apply_intervention(seq, best_coords[:, 0], best_coords[:, 1])
        interv_pred = model(seq_interv).squeeze()

        # Diferença Térmica em °C
        cooling_map = (base_pred - interv_pred).cpu().numpy() * std_lst

    # Prepara dados originais para plot (desnormaliza NDVI aproximado)
    ndvi_base = seq[0, CH_NDVI, :, :].cpu().numpy()

    plt.figure(figsize=(18, 5))

    # 1. Mapa de NDVI Base
    plt.subplot(1, 3, 1)
    plt.imshow(ndvi_base, cmap='YlGn', vmin=-1.5, vmax=2.0)
    plt.title("NDVI Original (Áreas Verdes)")
    plt.axis('off')

    # 2. Locais de Plantio Otimizados
    plt.subplot(1, 3, 2)
    plt.imshow(ndvi_base, cmap='gray', alpha=0.5)
    plt.scatter(best_coords[:, 1].cpu(), best_coords[:, 0].cpu(), c='lime', s=10, label='Novas Árvores')
    plt.title(f"Plano de Plantio ({num_trees} árvores)")
    plt.legend()
    plt.axis('off')

    # 3. Mapa de Impacto Térmico (Resfriamento)
    plt.subplot(1, 3, 3)
    # Criamos um colormap focado em resfriamento (Azul)
    div_norm = mcolors.TwoSlopeNorm(vmin=-0.1, vcenter=0.0, vmax=np.max(cooling_map))
    im = plt.imshow(cooling_map, cmap='coolwarm_r', norm=div_norm)
    plt.title(f"Resfriamento Estimado\nImpacto Total: {total_cooling:.2f} °C")
    plt.colorbar(im, label="Queda de Temperatura (°C)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    executar_simulacao(
        model_path="best_model_fold_0.pt",
        data_root="/content/drive/MyDrive/heat_dataset_v3",
        num_trees=500  # Quantas árvores a prefeitura quer plantar?
    )

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

def analisar_residuos_espaciais(model_path, data_root, city_name):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Carregar Modelo e Stats
    checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
    model = UrbanHeatNet().to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    std_lst = float(checkpoint['std'][0].flatten()[0])

    # 2. Carregar a cidade escolhida
    dataset = UrbanHeatDataset(data_root, allowed_cities=[city_name],
                               stats=(checkpoint['mean'], checkpoint['std']), is_train=False)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    seq, target = next(iter(loader))
    seq, target = seq.to(DEVICE), target.to(DEVICE)

    with torch.no_grad():
        # AQUI ESTÁ A MUDANÇA: Cortamos o canal 5 (NDWI) e passamos apenas 0 a 4
        model_input = seq[:, :CH_NDWI, :, :]
        pred = model(model_input).squeeze()

    # 3. Conversão para Celsius
    real_celsius = target.squeeze().cpu().numpy() * std_lst
    pred_celsius = pred.cpu().numpy() * std_lst
    residuos = real_celsius - pred_celsius

    # Preparar a entrada (seq) para visualização:
    # O seq tem shape (1, 6, 400, 400). Transformamos para (400, 400, 6)
    img_input = seq.squeeze().cpu().permute(1, 2, 0).numpy()

    # --- CÁLCULO DE PERCENTIL PARA ESCALA JUSTA ---
    v_min = np.percentile(real_celsius, 2)
    v_max = np.percentile(real_celsius, 98)

    # 4. Plotagem em 5 painéis (Adicionado o Painel de Entrada)
    fig, axes = plt.subplots(1, 5, figsize=(30, 6))

    # Painel 1: Entrada (Selecionando apenas o canal 0 para visualização)
    axes[0].imshow(img_input[:, :, 0], cmap='gray')
    axes[0].set_title("Entrada (Canal 0)")
    axes[0].axis('off')

    # Painel 2: Real
    im1 = axes[1].imshow(real_celsius, cmap='coolwarm_r', vmin=v_min, vmax=v_max)
    axes[1].set_title("Real (Ground Truth) °C")
    plt.colorbar(im1, ax=axes[1])

    # Painel 3: Predição
    im2 = axes[2].imshow(pred_celsius, cmap='coolwarm_r', vmin=v_min, vmax=v_max)
    axes[2].set_title("Predição do Modelo (°C)")
    plt.colorbar(im2, ax=axes[2])

    # Painel 4: Erro (Residual)
    im3 = axes[3].imshow(residuos, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[3].set_title("Erro Espacial (Real - Predito)")
    plt.colorbar(im3, ax=axes[3])

    # Painel 5: Histograma
    axes[4].hist(residuos.flatten(), bins=60, color='darkblue', alpha=0.7)
    axes[4].set_title("Distribuição do Erro")
    axes[4].set_xlim(-1.0, 1.0)

    plt.tight_layout()
    plt.show()

    # Estatísticas Críticas
    mae = np.mean(np.abs(residuos))
    bias = np.mean(residuos)
    print(f"📊 Relatório de Erro para {city_name}:")
    print(f"MAE (Erro Médio Absoluto): {mae:.4f} °C")
    print(f"BIAS (Viés Médio): {bias:.4f} °C")

if __name__ == "__main__":
    analisar_residuos_espaciais(
        model_path="best_model_fold_0.pt",
        data_root="/content/drive/MyDrive/heat_dataset_v3",
        city_name="Jose_de_Freitas"
    )

In [ ]:
import os
import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
import cv2
import functools
from concurrent.futures import ProcessPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIGURAÇÃO GERAL
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

SEQ_LEN = 2 # Quantidade de instantes de tempo no histórico
INPUT_CHANNELS = 8 # NDVI, NDWI, NDBI, DEM, LST_past, Coord_X, Coord_Y, Global_Mean_LST

CH_NDVI, CH_NDWI, CH_NDBI, CH_DEM, CH_LST = 0, 1, 2, 3, 4

BATCH_SIZE = 8 # Reduzido devido à adição da dimensão temporal
ACCUM_STEPS = 2
EPOCHS = 40
LR = 3e-4
WEIGHT_DECAY = 1e-4
K_FOLDS = 5
CROP_SIZE = 256

LST_CLIP_MIN, LST_CLIP_MAX = -20.0, 60.0

def ensure_channels_first(img):
    if img.ndim != 3: raise ValueError(f"Esperado array 3D, recebido {img.shape}")
    if img.shape[0] == 5: return img
    elif img.shape[-1] == 5: return np.transpose(img, (2, 0, 1))
    else: raise ValueError("Não foi possível identificar eixo de canais.")

# ============================================================
# FILTRO DE NUVENS
# ============================================================
def _check_cloud_fraction(filepath, temp_threshold, min_cloud_fraction):
    try:
        data = np.load(filepath)
        lst_band = ensure_channels_first(data)[CH_LST]
        cloud_fraction = np.sum(lst_band < temp_threshold) / lst_band.size
        return Path(filepath), cloud_fraction < min_cloud_fraction
    except Exception:
        return Path(filepath), False

def filter_cloudy_files_runtime(filepaths, temp_threshold=15.0, min_cloud_pct=30.0):
    valid_files = set()
    print(f"\n☁️ Filtrando {len(filepaths)} imagens...")
    check_func = functools.partial(_check_cloud_fraction, temp_threshold=temp_threshold, min_cloud_fraction=min_cloud_pct/100.0)
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(check_func, filepaths))
    for filepath, is_valid in results:
        if is_valid: valid_files.add(filepath)
    print(f" -> Imagens válidas mantidas: {len(valid_files)}\n")
    return valid_files

# ============================================================
# DATASET TEMPORAL COM CORREÇÃO FÍSICA E COORDCONV
# ============================================================
class UrbanHeatTemporalDataset(Dataset):
    def __init__(self, root, allowed_cities, valid_files_set, is_train=False):
        self.root = Path(root)
        self.is_train = is_train
        self.samples = []

        for city in allowed_cities:
            folder = self.root / city
            if not folder.exists(): continue
            files = sorted([f for f in folder.glob("*.npy") if f in valid_files_set])

            # Necessário SEQ_LEN imagens para entrada + 1 para o alvo
            if len(files) < SEQ_LEN + 1: continue

            for i in range(len(files) - SEQ_LEN):
                # (Lista de entradas, Arquivo Alvo)
                seq_files = files[i : i + SEQ_LEN]
                target_file = files[i + SEQ_LEN]
                self.samples.append((seq_files, target_file))

        # Criação dos grids de coordenadas para o CoordConv
        y_grid, x_grid = np.mgrid[-1:1:complex(0, CROP_SIZE), -1:1:complex(0, CROP_SIZE)]
        self.coord_y = y_grid.astype(np.float32)
        self.coord_x = x_grid.astype(np.float32)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        seq_files, target_file = self.samples[idx]

        # 1. Carrega as imagens completas
        seq_imgs = [ensure_channels_first(np.load(f).astype(np.float32)) for f in seq_files]
        tgt_img = ensure_channels_first(np.load(target_file).astype(np.float32))

        # 2. Definição do Crop Central ou Aleatório
        max_h, max_w = tgt_img.shape[1], tgt_img.shape[2]
        if self.is_train and max_h > CROP_SIZE and max_w > CROP_SIZE:
            y, x = random.randint(0, max_h - CROP_SIZE), random.randint(0, max_w - CROP_SIZE)
        else:
            y, x = max(0, (max_h - CROP_SIZE) // 2), max(0, (max_w - CROP_SIZE) // 2)

        # 3. Tratamento do Alvo (Anomalia baseada na CIDADE INTEIRA)
        is_nodata_tgt = (tgt_img[CH_NDVI] == 0.0) & (tgt_img[CH_NDBI] == 0.0)
        valid_mask_tgt_full = ~is_nodata_tgt

        valid_pixels_tgt = tgt_img[CH_LST][valid_mask_tgt_full]
        city_mean_target = valid_pixels_tgt.mean() if len(valid_pixels_tgt) > 0 else 0.0

        # Anomalia Global calculada antes do crop
        tgt_anomaly_full = (tgt_img[CH_LST] - city_mean_target) * valid_mask_tgt_full.astype(np.float32)

        cropped_tgt_anomaly = tgt_anomaly_full[y:y+CROP_SIZE, x:x+CROP_SIZE]
        crop_valid_mask = valid_mask_tgt_full[y:y+CROP_SIZE, x:x+CROP_SIZE].astype(np.float32)

        padded_tgt_anomaly = np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.float32)
        valid_mask = np.zeros((1, CROP_SIZE, CROP_SIZE), dtype=np.float32)
        padded_tgt_anomaly[:cropped_tgt_anomaly.shape[0], :cropped_tgt_anomaly.shape[1]] = cropped_tgt_anomaly
        valid_mask[0, :crop_valid_mask.shape[0], :crop_valid_mask.shape[1]] = crop_valid_mask

        target_norm = (np.clip(padded_tgt_anomaly, -20.0, 20.0) / 5.0) * valid_mask[0]

        # 4. Construção da Sequência (Histórico)
        seq_tensors = []
        for img_t in seq_imgs:
            # Calcula a média da cidade no instante t
            mask_t = ~((img_t[CH_NDVI] == 0.0) & (img_t[CH_NDBI] == 0.0))
            city_mean_t = img_t[CH_LST][mask_t].mean() if mask_t.sum() > 0 else 0.0
            anomaly_t = (img_t[CH_LST] - city_mean_t) * mask_t.astype(np.float32)

            cropped_base = img_t[:, y:y+CROP_SIZE, x:x+CROP_SIZE]
            cropped_anomaly_t = anomaly_t[y:y+CROP_SIZE, x:x+CROP_SIZE]

            padded_base = np.zeros((5, CROP_SIZE, CROP_SIZE), dtype=np.float32)
            padded_base[:, :cropped_base.shape[1], :cropped_base.shape[2]] = cropped_base

            padded_base[CH_NDVI] = np.clip(padded_base[CH_NDVI], -1.0, 1.0)
            padded_base[CH_NDWI] = np.clip(padded_base[CH_NDWI], -1.0, 1.0)
            padded_base[CH_NDBI] = np.clip(padded_base[CH_NDBI], -1.0, 1.0)

            if valid_mask.sum() > 0:
                local_dem_mean = padded_base[CH_DEM][valid_mask[0] == 1.0].mean()
                padded_base[CH_DEM] = (padded_base[CH_DEM] - local_dem_mean) / 50.0

            # Anomalia LST do passado normalizada
            lst_past_norm = np.clip(cropped_anomaly_t, -20.0, 20.0) / 5.0
            lst_past_padded = np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.float32)
            lst_past_padded[:lst_past_norm.shape[0], :lst_past_norm.shape[1]] = lst_past_norm

            # Global Mean (Proxy Meteorológico) expandido espacialmente
            global_mean_layer = np.full((CROP_SIZE, CROP_SIZE), city_mean_t / 40.0, dtype=np.float32)

            # Empilha: [NDVI, NDWI, NDBI, DEM, LST_past_anomaly, CoordY, CoordX, GlobalMean]
            step_features = np.stack([
                padded_base[CH_NDVI] * valid_mask[0],
                padded_base[CH_NDWI] * valid_mask[0],
                padded_base[CH_NDBI] * valid_mask[0],
                padded_base[CH_DEM] * valid_mask[0],
                lst_past_padded * valid_mask[0],
                self.coord_y * valid_mask[0],
                self.coord_x * valid_mask[0],
                global_mean_layer * valid_mask[0]
            ], axis=0)

            seq_tensors.append(torch.from_numpy(step_features).float())

        # seq_out shape: (Time, Channels, H, W)
        seq_out = torch.stack(seq_tensors, dim=0)
        out_tgt = torch.from_numpy(target_norm).unsqueeze(0).float()
        out_valid = torch.from_numpy(valid_mask).float()

        # Augmentation (aplicado consistentemente em todo o volume de tempo)
        if self.is_train:
            if random.random() > 0.5:
                seq_out = torch.flip(seq_out, dims=[-1])
                out_tgt, out_valid = torch.flip(out_tgt, dims=[-1]), torch.flip(out_valid, dims=[-1])
            if random.random() > 0.5:
                seq_out = torch.flip(seq_out, dims=[-2])
                out_tgt, out_valid = torch.flip(out_tgt, dims=[-2]), torch.flip(out_valid, dims=[-2])
            k = random.randint(0, 3)
            if k > 0:
                seq_out = torch.rot90(seq_out, k, dims=[-2, -1])
                out_tgt, out_valid = torch.rot90(out_tgt, k, dims=[-2, -1]), torch.rot90(out_valid, k, dims=[-2, -1])

        return seq_out, out_tgt, out_valid, city_mean_target

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels)
        )
        self.act = nn.GELU()

        self.shortcut = nn.Identity()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )

    def forward(self, x):
        out = self.conv2(self.conv1(x))
        return self.act(out + self.shortcut(x))

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.GroupNorm(8, F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.GroupNorm(8, F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.act = nn.GELU()

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)

        psi_out = self.psi(self.act(g1 + x1))
        return x * psi_out

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = ResidualBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x_up = self.up(x)
        return self.conv(torch.cat([x_up, skip], dim=1))

class HybridUrbanHeatNet(nn.Module):
    def __init__(self, in_channels=4, base_c=64):
        super().__init__()
        self.stem = ResidualBlock(in_channels, base_c)

        # Encoder
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c, base_c*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c*2, base_c*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c*4, base_c*8))

        # Bottleneck estruturado exatamente como o checkpoint espera:
        # - conv1: ConvBNAct (gera chaves bottle.conv1.block.*)
        # - conv2: Sequential direto (gera chaves bottle.conv2.0.* e bottle.conv2.1.*)
        self.bottle = nn.ModuleDict({
            'conv1': ConvBNAct(base_c*8, base_c*8),
            'conv2': nn.Sequential(
                nn.Conv2d(base_c*8, base_c*8, kernel_size=3, padding=1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=base_c*8)
            )
        })

        # Decoder + Attention Gates
        self.att3 = AttentionBlock(base_c*8, base_c*4, base_c*4)
        self.up3 = UpBlock(base_c*8, base_c*4, base_c*4)

        self.att2 = AttentionBlock(base_c*4, base_c*2, base_c*2)
        self.up2 = UpBlock(base_c*4, base_c*2, base_c*2)

        self.att1 = AttentionBlock(base_c*2, base_c, base_c)
        self.up1 = UpBlock(base_c*2, base_c, base_c)

        # Head
        self.head = nn.Sequential(
            nn.Conv2d(base_c, base_c//2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(base_c//2, 1, 1)
        )

    def forward(self, x):
        s1 = self.stem(x)
        s2 = self.down1(s1)
        s3 = self.down2(s2)

        # Bottleneck forward
        d3 = self.down3(s3)
        b_feat = self.bottle['conv1'](d3)
        bottle = self.bottle['conv2'](b_feat)

        u3 = self.up3(bottle, self.att3(bottle, s3))
        u2 = self.up2(u3, self.att2(u3, s2))
        u1 = self.up1(u2, self.att1(u2, s1))

        return self.head(u1)

class BaselineLoss(nn.Module):
    def __init__(self, hotspot_w=2.0, top_q=0.9):
        super().__init__()
        self.hotspot_w = hotspot_w
        self.top_q = top_q

    def forward(self, pred, target, valid_mask):
        pred = pred * valid_mask
        target = target * valid_mask

        l1_diff = torch.abs(pred - target)
        valid_count = valid_mask.sum() + 1e-8
        base_loss = l1_diff.sum() / valid_count

        hotspot_mask = torch.zeros_like(target)
        for i in range(target.size(0)):
            valid_pixels = target[i][valid_mask[i] == 1.0]
            if valid_pixels.numel() > 0:
                q90 = torch.quantile(valid_pixels, self.top_q)
                hotspot_mask[i] = (target[i] >= q90).float() * valid_mask[i]

        hs_count = hotspot_mask.sum() + 1e-8
        hotspot_loss = (l1_diff * hotspot_mask).sum() / hs_count if hotspot_mask.sum() > 0 else torch.tensor(0.0, device=target.device)

        return base_loss + (self.hotspot_w * hotspot_loss)

# ============================================================
# INTERPRETABILIDADE: GRAD-CAM PARA CONV-LSTM
# ============================================================
class RegressionGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def generate(self, input_tensor):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output.sum()
        score.backward()

        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().detach().numpy()

        if cam.ndim == 3: cam = cam[0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output.squeeze().cpu().detach().numpy()

def plot_grad_cam(input_tensor, cam_map, pred_norm, real_norm, city_mean, save_path="grad_cam.png"):
    # Pegamos o NDVI do último instante de tempo do histórico (t-1)
    ndvi_input = input_tensor[0, -1, CH_NDVI].cpu().numpy()

    # Reconstrução LST Absoluta usando a média correta do Batch/Crop
    pred_celsius = (pred_norm * 5.0) + city_mean
    real_celsius = (real_norm * 5.0) + city_mean

    if pred_celsius.ndim == 3: pred_celsius = pred_celsius[0]
    if real_celsius.ndim == 3: real_celsius = real_celsius[0]

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(ndvi_input, cmap='gray')
    axes[0].set_title('NDVI (Instante Atual)')

    heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    axes[1].imshow(ndvi_input, cmap='gray', alpha=0.5)
    axes[1].imshow(heatmap, alpha=0.6)
    axes[1].set_title('Atenção Espaço-Temporal')

    im2 = axes[2].imshow(pred_celsius, cmap='inferno')
    axes[2].set_title('Temp. Prevista (°C)')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    im3 = axes[3].imshow(real_celsius, cmap='inferno')
    axes[3].set_title('Temp. Real (°C)')
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================
if __name__ == "__main__":
    all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]
    all_files_raw = list(Path(DATA_DIR).rglob("*.npy"))
    valid_files_set = filter_cloudy_files_runtime(all_files_raw)

    print("\n2. Iniciando K-Fold Cross Validation Espaço-Temporal...")
    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    folds_list = list(kf.split(all_cities))

    for fold, (train_idx, val_idx) in enumerate(folds_list):
        train_cities = [all_cities[i] for i in train_idx]
        val_cities = [all_cities[i] for i in val_idx]

        train_ds = UrbanHeatTemporalDataset(DATA_DIR, allowed_cities=train_cities, valid_files_set=valid_files_set, is_train=True)
        val_ds = UrbanHeatTemporalDataset(DATA_DIR, allowed_cities=val_cities, valid_files_set=valid_files_set, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = HybridUrbanHeatNet(in_channels=INPUT_CHANNELS).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=6)

        loss_fn = BaselineLoss()
        best_loss = float('inf')

        print(f"\n--- Iniciando FOLD {fold + 1} ---")
        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0
            optimizer.zero_grad()

            for step, (seq, target, valid, city_mean) in enumerate(train_loader):
                seq, target, valid = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE)

                pred = model(seq)
                loss = loss_fn(pred, target, valid)
                (loss / ACCUM_STEPS).backward()
                train_loss += loss.item()

                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

            model.eval()
            val_loss, val_mae_c, val_hs_mae_c, val_iou = 0, 0, 0, 0

            with torch.no_grad():
                for seq, target, valid, city_mean in val_loader:
                    seq, target, valid = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE)
                    pred = model(seq)

                    val_loss += loss_fn(pred, target, valid).item()

                    # Passando o city_mean do batch para reconstrução térmica correta
                    mae_c, hs_mae_c, iou = compute_metrics(pred, target, valid, city_mean.to(DEVICE), top_q=0.9)
                    val_mae_c += mae_c
                    val_hs_mae_c += hs_mae_c
                    val_iou += iou

            n_t, n_v = max(len(train_loader), 1), max(len(val_loader), 1)
            train_loss /= n_t
            val_loss /= n_v
            scheduler.step(val_loss)

            print(f"Epoch {epoch+1:02d} | T.Loss: {train_loss:.4f} | V.Loss: {val_loss:.4f} | MAE (°C): {val_mae_c/n_v:.4f} | HS MAE: {val_hs_mae_c/n_v:.4f} | IoU: {val_iou/n_v:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(model.state_dict(), f"best_convlstm_fold_{fold}.pt")

        # =======================================================
        # GERAÇÃO DO GRAD-CAM NO FIM DO FOLD
        # =======================================================
        print(f"\nGerando Grad-CAM Espaço-Temporal para o Fold {fold+1}...")

        model.load_state_dict(torch.load(f"best_convlstm_fold_{fold}.pt", weights_only=True))
        cam_generator = RegressionGradCAM(model, target_layer=model.ups[-1])

        sample_seq, sample_tgt, sample_valid, sample_city_mean = next(iter(val_loader))

        cam_map, pred_np = cam_generator.generate(sample_seq[0:1].to(DEVICE))
        real_np = sample_tgt[0:1].cpu().numpy()
        city_mean_val = sample_city_mean[0].item()

        plot_grad_cam(
            sample_seq[0:1],
            cam_map,
            pred_np,
            real_np,
            city_mean_val,
            save_path=f"gradcam_convlstm_fold_{fold+1}.png"
        )
        print(f"-> Imagem do Grad-CAM salva como 'gradcam_convlstm_fold_{fold+1}.png'\n")

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from torch.utils.data import DataLoader

def analisar_residuos_e_plantio_hibrido(model_path, data_root, city_name):
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Aplicar o Filtro de Nuvens específico para a cidade escolhida
    city_path = Path(data_root) / city_name
    all_city_files = list(city_path.glob("*.npy"))

    print(f"\n🔍 Analisando {city_name}...")
    valid_files_set = filter_cloudy_files_runtime(all_city_files, temp_threshold=15.0, min_cloud_pct=30.0)

    if len(valid_files_set) < 2:
        print(f"❌ Imagens válidas insuficientes para {city_name} após o filtro de nuvens.")
        return

    # 2. Computar estatísticas locais
    # Como o novo torch.save guarda apenas os pesos, recalculamos as médias para a cidade
    print("📊 Calculando estatísticas de normalização locais...")
    mean_city, std_city = compute_dataset_statistics(valid_files_set)
    global_stats = (mean_city, std_city)

    # 3. Carregar o Modelo Híbrido
    model = HybridUrbanHeatNet().to(DEVICE)
    # O novo script salva apenas o state_dict diretamente
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    # 4. Carregar o Dataset
    dataset = UrbanHeatDataset(
        data_root,
        allowed_cities=[city_name],
        valid_files_set=valid_files_set,
        stats=global_stats,
        is_train=False
    )
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    # O novo dataset retorna 3 elementos
    seq, target, valid_mask = next(iter(loader))
    seq, target, valid_mask = seq.to(DEVICE), target.to(DEVICE), valid_mask.to(DEVICE)

    # 5. Inferência Base
    with torch.no_grad():
        model_input = seq[:, :5, :, :]
        pred_base = model(model_input).squeeze()

    # 6. Otimização com Algoritmo Genético
    print(f"🌲 Iniciando simulação de plantio com Algoritmo Genético...")
    simulator = GeneticPlantingSimulator(model, DEVICE, global_stats)

    opt_y, opt_x, best_fitness = simulator.optimize_genetic(
        seq, num_trees=50, pop_size=30, generations=50
    )

    # 7. Inferência com Intervenção (Cenário Pós-Plantio)
    with torch.no_grad():
        y_tensor = torch.tensor(opt_y, device=DEVICE)
        x_tensor = torch.tensor(opt_x, device=DEVICE)
        seq_interv = simulator.apply_intervention(seq, y_tensor, x_tensor)
        pred_interv = model(seq_interv[:, :5, :, :]).squeeze()

    # 8. Conversão (O modelo prediz a Anomalia Espacial Normalizada Z-Score)
    real_anomaly = target.squeeze().cpu().numpy()
    pred_anomaly = pred_base.cpu().numpy()
    pred_interv_anomaly = pred_interv.cpu().numpy()

    residuos = real_anomaly - pred_anomaly
    cooling_map = pred_anomaly - pred_interv_anomaly # Valores positivos = resfriamento

    # Preparar a entrada para visualização (foco no NDVI - Canal 0)
    img_input = seq.squeeze().cpu().permute(1, 2, 0).numpy()
    ndvi_map = img_input[:, :, 0]

    # Limites para escala de cores justas ignorando outliers
    v_min = np.percentile(real_anomaly, 2)
    v_max = np.percentile(real_anomaly, 98)

    # 9. Plotagem em Grade 2x3
    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    axes = axes.flatten()

    # Painel 1: Entrada (NDVI)
    axes[0].imshow(ndvi_map, cmap='Greens')
    axes[0].set_title("NDVI Original (Vegetação Atual)")
    axes[0].axis('off')

    # Painel 2: Real
    im1 = axes[1].imshow(real_anomaly, cmap='coolwarm_r', vmin=v_min, vmax=v_max)
    axes[1].set_title("Real (Anomalia Z-Score)")
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    axes[1].axis('off')

    # Painel 3: Predição Base
    im2 = axes[2].imshow(pred_anomaly, cmap='coolwarm_r', vmin=v_min, vmax=v_max)
    axes[2].set_title("Predição da HybridNet")
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    axes[2].axis('off')

    # Painel 4: Erro (Residual)
    im3 = axes[3].imshow(residuos, cmap='RdBu_r', vmin=-0.5, vmax=0.5)
    axes[3].set_title("Erro Espacial (Real - Predito)")
    plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)
    axes[3].axis('off')

    # Painel 5: Locais de Plantio Otimizados
    axes[4].imshow(ndvi_map, cmap='Greens')
    axes[4].scatter(opt_x, opt_y, c='red', s=40, marker='X', label='Árvores Sugeridas')
    axes[4].set_title(f"Locais de Plantio ({len(opt_x)} Árvores)")
    axes[4].legend(loc="upper right")
    axes[4].axis('off')

    # Painel 6: Mapa de Resfriamento
    v_cool_max = max(np.max(cooling_map), 0.01)
    im5 = axes[5].imshow(cooling_map, cmap='Blues', vmin=0, vmax=v_cool_max)
    axes[5].set_title("Impacto do Plantio (Redução da Anomalia)")
    plt.colorbar(im5, ax=axes[5], fraction=0.046, pad=0.04)
    axes[5].axis('off')

    plt.tight_layout()
    plt.show()

    # Estatísticas Críticas considerando a valid_mask
    valid_mask_np = valid_mask.squeeze().cpu().numpy()
    mae = np.sum(np.abs(residuos) * valid_mask_np) / (np.sum(valid_mask_np) + 1e-8)
    bias = np.sum(residuos * valid_mask_np) / (np.sum(valid_mask_np) + 1e-8)
    resfriamento_total = cooling_map[cooling_map > 0].sum()

    print(f"\n📊 Relatório de Desempenho para {city_name}:")
    print(f"MAE (Erro Médio Absoluto em área válida): {mae:.4f}")
    print(f"BIAS (Viés Médio em área válida): {bias:.4f}")
    print(f"🌡️ Resfriamento Total Estimado (Z-Score): {resfriamento_total:.2f}")

if __name__ == "__main__":
    analisar_residuos_e_plantio_hibrido(
        model_path="best_hybrid_fold_0.pt",
        data_root="/content/drive/MyDrive/heat_dataset_v3",
        city_name="Teresina" # Teste com outra cidade se quiser
    )

In [ ]:
from google.colab import files

# Caminho do arquivo
arquivo = "best_hybrid_fold_0.pt"

# Faz o download
files.download(arquivo)

In [ ]:
import os
import random
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import math
import matplotlib.pyplot as plt
import cv2
import functools
from concurrent.futures import ProcessPoolExecutor

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIGURAÇÃO GERAL
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "/content/drive/MyDrive/heat_dataset_v3"

TOTAL_RAW_CHANNELS = 5
CHANNELS_TO_KEEP = [0, 1, 2, 3, 4]
INPUT_CHANNELS = len(CHANNELS_TO_KEEP)

CH_NDVI, CH_NDWI, CH_NDBI, CH_DEM, CH_LST = 0, 1, 2, 3, 4

BATCH_SIZE = 2
EPOCHS = 30
LR = 5e-4
WEIGHT_DECAY = 1e-4
K_FOLDS = 5

LST_CLIP_MIN, LST_CLIP_MAX = -20.0, 60.0

def ensure_channels_first(img):
    if img.ndim != 3: raise ValueError(f"Esperado array 3D, recebido {img.shape}")
    if img.shape[0] == TOTAL_RAW_CHANNELS: return img
    elif img.shape[-1] == TOTAL_RAW_CHANNELS: return np.transpose(img, (2, 0, 1))
    else: raise ValueError("Não foi possível identificar eixo de canais.")

# ============================================================
# FILTRO DE NUVENS EM PARALELO
# ============================================================
def _check_cloud_fraction(filepath, temp_threshold, min_cloud_fraction):
    try:
        data = np.load(filepath)
        lst_band = ensure_channels_first(data)[CH_LST]
        cloud_fraction = np.sum(lst_band < temp_threshold) / lst_band.size
        is_valid = cloud_fraction < min_cloud_fraction
        return Path(filepath), is_valid
    except Exception:
        return Path(filepath), False

def filter_cloudy_files_runtime(filepaths, temp_threshold=15.0, min_cloud_pct=30.0):
    min_cloud_fraction = min_cloud_pct / 100.0
    valid_files = set()
    discarded_count = 0

    print(f"\n☁️ Filtrando {len(filepaths)} imagens (Limite de Nuvem: >{min_cloud_pct}%) em paralelo...")

    check_func = functools.partial(_check_cloud_fraction,
                                   temp_threshold=temp_threshold,
                                   min_cloud_fraction=min_cloud_fraction)

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(check_func, filepaths))

    for filepath, is_valid in results:
        if is_valid:
            valid_files.add(filepath)
        else:
            discarded_count += 1

    print(f" -> Imagens descartadas (Muito nubladas): {discarded_count}")
    print(f" -> Imagens válidas mantidas: {len(valid_files)}\n")

    return valid_files

# ============================================================
# ESTATÍSTICAS E NORMALIZAÇÃO
# ============================================================
def compute_dataset_statistics(files):
    count = 0
    sum_ = np.zeros(INPUT_CHANNELS, dtype=np.float64)
    sum_sq = np.zeros(INPUT_CHANNELS, dtype=np.float64)
    for f in files:
        img = ensure_channels_first(np.load(f).astype(np.float32))[CHANNELS_TO_KEEP]
        img[CH_LST] = np.clip(img[CH_LST], LST_CLIP_MIN, LST_CLIP_MAX)
        img[CH_NDVI] = np.clip(img[CH_NDVI], -1.0, 1.0)
        img[CH_NDWI] = np.clip(img[CH_NDWI], -1.0, 1.0)
        flat = img.reshape(INPUT_CHANNELS, -1).astype(np.float64)
        sum_ += flat.sum(axis=1)
        sum_sq += (flat ** 2).sum(axis=1)
        count += flat.shape[1]
    mean = sum_ / count
    var = np.clip(sum_sq / count - mean ** 2, a_min=0.0, a_max=None)
    return mean.astype(np.float32), np.sqrt(var).astype(np.float32)

class Normalize:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean, dtype=torch.float32).view(INPUT_CHANNELS, 1, 1)
        self.std = torch.tensor(std, dtype=torch.float32).view(INPUT_CHANNELS, 1, 1) + 1e-6
    def __call__(self, img):
        if isinstance(img, np.ndarray): img = torch.from_numpy(img).float()
        return (img - self.mean) / self.std

# ============================================================
# DATASET
# ============================================================
class UrbanHeatDataset(Dataset):
    def __init__(self, root, allowed_cities, valid_files_set, normalize=True, stats=None, is_train=False):
        self.root = Path(root)
        self.normalize = normalize
        self.is_train = is_train
        self.samples = []
        all_files = []

        for city in allowed_cities:
            folder = self.root / city
            if not folder.exists(): continue

            files = sorted([f for f in folder.glob("*.npy") if f in valid_files_set])
            all_files.extend(files)

            if len(files) < 2: continue
            for i in range(len(files) - 1):
                self.samples.append((files[i], files[i + 1]))

        if normalize and len(all_files) > 0:
            if stats is not None:
                self.mean, self.std = stats
            else:
                self.mean, self.std = compute_dataset_statistics(all_files)
            self.normalizer = Normalize(self.mean, self.std)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        base_file, target_file = self.samples[idx]
        base_img = ensure_channels_first(np.load(base_file).astype(np.float32))[CHANNELS_TO_KEEP]
        tgt_img = ensure_channels_first(np.load(target_file).astype(np.float32))[CHANNELS_TO_KEEP]

        max_h = min(base_img.shape[1], tgt_img.shape[1])
        max_w = min(base_img.shape[2], tgt_img.shape[2])
        crop_size = 400

        if self.is_train and max_h > crop_size and max_w > crop_size:
            y = random.randint(0, max_h - crop_size)
            x = random.randint(0, max_w - crop_size)
        else:
            y, x = max(0, (max_h - crop_size) // 2), max(0, (max_w - crop_size) // 2)

        cropped_base = base_img[:, y:y+crop_size, x:x+crop_size]
        padded_base = np.zeros((INPUT_CHANNELS, crop_size, crop_size), dtype=np.float32)
        padded_base[:, :cropped_base.shape[1], :cropped_base.shape[2]] = cropped_base

        valid_mask = np.zeros((1, crop_size, crop_size), dtype=np.float32)
        valid_mask[0, :cropped_base.shape[1], :cropped_base.shape[2]] = 1.0

        padded_base[CH_LST] = np.clip(padded_base[CH_LST], LST_CLIP_MIN, LST_CLIP_MAX)
        padded_base[CH_NDVI] = np.clip(padded_base[CH_NDVI], -1.0, 1.0)
        padded_base[CH_NDWI] = np.clip(padded_base[CH_NDWI], -1.0, 1.0)

        if self.normalize:
            padded_base = self.normalizer(padded_base)
            if isinstance(padded_base, torch.Tensor):
                padded_base = padded_base.numpy()

        padded_base = padded_base * valid_mask

        if isinstance(padded_base, np.ndarray):
            padded_base = torch.from_numpy(padded_base).float()

        model_features = padded_base[[CH_NDVI, CH_NDBI, CH_DEM], :, :]
        water_mask = padded_base[[CH_NDWI], :, :]
        out_seq = torch.cat([model_features, water_mask], dim=0)

        cropped_tgt = tgt_img[CH_LST, y:y+crop_size, x:x+crop_size]
        padded_tgt_lst = np.zeros((crop_size, crop_size), dtype=np.float32)
        padded_tgt_lst[:cropped_tgt.shape[0], :cropped_tgt.shape[1]] = cropped_tgt
        padded_tgt_lst = np.clip(padded_tgt_lst, LST_CLIP_MIN, LST_CLIP_MAX)

        tgt_tensor = torch.from_numpy(padded_tgt_lst)
        valid_mask_tensor = torch.from_numpy(valid_mask[0])

        valid_tgt = tgt_tensor[valid_mask_tensor == 1.0]
        macro_climate = valid_tgt.mean() if valid_tgt.numel() > 0 else tgt_tensor.mean()

        if valid_tgt.numel() > 1:
            std_anomaly = valid_tgt.std() + 1e-6
        else:
            std_anomaly = torch.tensor(1.0)

        spatial_anomaly = (tgt_tensor - macro_climate) * valid_mask_tensor
        out_tgt = (spatial_anomaly / std_anomaly).unsqueeze(0).float()
        out_valid = torch.tensor(valid_mask).float()

        # ========================================================
        # DATA AUGMENTATION (Apenas no treino para generalização)
        # ========================================================
        if self.is_train:
            # Flip Horizontal
            if random.random() > 0.5:
                out_seq = torch.flip(out_seq, dims=[-1])
                out_tgt = torch.flip(out_tgt, dims=[-1])
                out_valid = torch.flip(out_valid, dims=[-1])

            # Flip Vertical
            if random.random() > 0.5:
                out_seq = torch.flip(out_seq, dims=[-2])
                out_tgt = torch.flip(out_tgt, dims=[-2])
                out_valid = torch.flip(out_valid, dims=[-2])

            # Rotação 90/180/270
            k = random.randint(0, 3)
            if k > 0:
                out_seq = torch.rot90(out_seq, k, dims=[-2, -1])
                out_tgt = torch.rot90(out_tgt, k, dims=[-2, -1])
                out_valid = torch.rot90(out_valid, k, dims=[-2, -1])

        return out_seq, out_tgt, out_valid, std_anomaly.clone().detach().float()

# ============================================================
# BLOCOS CONSTRUTIVOS
# ============================================================
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, padding=1, dilation=1, kernel_size=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, dilation=dilation, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_ch),
            nn.GELU()
        )
    def forward(self, x): return self.block(x)

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = ConvBNAct(in_channels, out_channels)
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(num_groups=8, num_channels=out_channels)
        )
        self.act = nn.GELU()
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
    def forward(self, x):
        return self.act(self.conv2(self.conv1(x)) + self.shortcut(x))

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=False), nn.GroupNorm(8, F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=False), nn.GroupNorm(8, F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=False), nn.BatchNorm2d(1), nn.Sigmoid())
        self.act = nn.GELU()

    def forward(self, g, x):
        g1, x1 = self.W_g(g), self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        psi = self.act(g1 + x1)
        return x * self.psi(psi)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.conv = ResidualBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        return self.conv(torch.cat([self.up(x), skip], dim=1))

class HybridUrbanHeatNet(nn.Module):
    def __init__(self, in_channels=4, base_c=64):
        super().__init__()
        self.stem = ResidualBlock(in_channels, base_c)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c, base_c*2))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c*2, base_c*4))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), ResidualBlock(base_c*4, base_c*8))

        # Gargalo baseado em ResidualBlock puro (sem ASPP)
        self.bottle = ResidualBlock(base_c*8, base_c*8)

        self.att3 = AttentionBlock(F_g=base_c*8, F_l=base_c*4, F_int=base_c*4)
        self.up3 = UpBlock(base_c*8, base_c*4, base_c*4)

        self.att2 = AttentionBlock(F_g=base_c*4, F_l=base_c*2, F_int=base_c*2)
        self.up2 = UpBlock(base_c*4, base_c*2, base_c*2)

        self.att1 = AttentionBlock(F_g=base_c*2, F_l=base_c, F_int=base_c)
        self.up1 = UpBlock(base_c*2, base_c, base_c)

        self.head = nn.Sequential(
            nn.Conv2d(base_c, base_c//2, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(base_c//2, 1, 1)
        )

    def forward(self, x):
        s1 = self.stem(x)
        s2 = self.down1(s1)
        s3 = self.down2(s2)
        d_deep = self.down3(s3)

        bottle = self.bottle(d_deep)

        x3 = self.att3(g=bottle, x=s3)
        u3 = self.up3(bottle, x3)

        x2 = self.att2(g=u3, x=s2)
        u2 = self.up2(u3, x2)

        x1 = self.att1(g=u2, x=s1)
        u1 = self.up1(u2, x1)

        return self.head(u1)

# ============================================================
# MÉTRICAS E LOSS
# ============================================================
def compute_metrics(pred, target, valid_mask, std_anomaly, top_q=0.9):
    std_expanded = std_anomaly.view(-1, 1, 1, 1)
    pred_c = pred * std_expanded
    target_c = target * std_expanded

    valid_count = valid_mask.sum() + 1e-8
    mae_celsius = (torch.abs(pred_c - target_c) * valid_mask).sum() / valid_count

    batch_size = target.size(0)
    intersection = 0.0
    union = 0.0
    hotspot_mae = 0.0
    hs_count = 0.0

    for i in range(batch_size):
        valid_pixels = target_c[i][valid_mask[i] == 1.0]
        if valid_pixels.numel() > 0:
            q90_real = torch.quantile(valid_pixels, top_q)

            hs_real_mask = (target_c[i] >= q90_real) & (valid_mask[i] == 1.0)
            hs_pred_mask = (pred_c[i] >= q90_real) & (valid_mask[i] == 1.0)

            intersection += (hs_pred_mask & hs_real_mask).float().sum()
            union += (hs_pred_mask | hs_real_mask).float().sum()

            if hs_real_mask.sum() > 0:
                hotspot_mae += (torch.abs(pred_c[i] - target_c[i]) * hs_real_mask.float()).sum()
                hs_count += hs_real_mask.float().sum()

    iou = (intersection + 1e-6) / (union + 1e-6)
    hotspot_mae_celsius = hotspot_mae / (hs_count + 1e-8)

    return mae_celsius.item(), hotspot_mae_celsius.item(), iou.item()

def create_gaussian_window(window_size, sigma=1.5):
    # Create a 1D Gaussian
    coords = torch.arange(window_size, dtype=torch.float32)
    coords -= window_size // 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g /= g.sum()

    # Create a 2D Gaussian by outer product
    g2d = g.unsqueeze(1) * g.unsqueeze(0)
    # Reshape to [out_channels, in_channels/groups, H, W]
    window = g2d.unsqueeze(0).unsqueeze(0)
    return window

class SSIMLoss(nn.Module):
    def __init__(self, window_size=11, data_range=6.0):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.register_buffer('window', create_gaussian_window(window_size))

    def forward(self, pred, target, valid_mask):
        # Ensure window is on the same device as the inputs
        window = self.window.to(pred.device)

        # Calculate local means
        mu1 = F.conv2d(pred, window, padding=self.window_size//2)
        mu2 = F.conv2d(target, window, padding=self.window_size//2)

        mu1_sq = mu1 ** 2
        mu2_sq = mu2 ** 2
        mu1_mu2 = mu1 * mu2

        # Calculate local variances and covariances
        sigma1_sq = F.conv2d(pred * pred, window, padding=self.window_size//2) - mu1_sq
        sigma2_sq = F.conv2d(target * target, window, padding=self.window_size//2) - mu2_sq
        sigma12 = F.conv2d(pred * target, window, padding=self.window_size//2) - mu1_mu2

        # SSIM constants based on data range
        C1 = (0.01 * self.data_range) ** 2
        C2 = (0.03 * self.data_range) ** 2

        # SSIM formula
        ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

        # SSIM is 1 for perfect match. Loss is (1 - SSIM). Apply mask to ignore padding.
        loss_map = (1.0 - ssim_map) / 2.0

        valid_count = valid_mask.sum() + 1e-8
        return (loss_map * valid_mask).sum() / valid_count

class HybridPhysicsLoss(nn.Module):
    def __init__(self, ssim_w=0.8, l1_w=0.2, hotspot_w=0.1, top_q=0.9):
        super().__init__()
        self.l1_loss = nn.L1Loss(reduction='none')
        self.ssim_loss = SSIMLoss(window_size=11, data_range=6.0)

        self.ssim_w = ssim_w
        self.l1_w = l1_w
        self.hotspot_w = hotspot_w
        self.top_q = top_q

    def forward(self, pred, target, ndvi_map, valid_mask):
        valid_count = valid_mask.sum() + 1e-8

        # 1. L1 Base Loss (Anchors absolute values)
        l1_map = self.l1_loss(pred, target) * valid_mask
        l1_loss_val = l1_map.sum() / valid_count

        # 2. SSIM Loss (Forces structural patterns)
        ssim_loss_val = self.ssim_loss(pred, target, valid_mask)

        # 3. Hotspot Loss
        hotspot_mask = torch.zeros_like(target)
        for i in range(target.size(0)):
            valid_pixels = target[i][valid_mask[i] == 1.0]
            if valid_pixels.numel() > 0:
                q90 = torch.quantile(valid_pixels, self.top_q)
                hotspot_mask[i] = (target[i] >= q90).float() * valid_mask[i]

        if hotspot_mask.sum() > 0:
            hotspot_loss_val = (l1_map * hotspot_mask).sum() / hotspot_mask.sum()
        else:
            hotspot_loss_val = torch.tensor(0.0, device=target.device)

        # Final Combined Loss
        total_loss = (self.ssim_w * ssim_loss_val) + (self.l1_w * l1_loss_val) + (self.hotspot_w * hotspot_loss_val)

        return total_loss

# ============================================================
# GRAD-CAM & OVERFIT
# ============================================================
class RegressionGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output): self.activations = output
    def save_gradient(self, module, grad_in, grad_out): self.gradients = grad_out[0]

    def generate(self, input_tensor):
        self.model.eval()
        self.model.zero_grad()
        output = self.model(input_tensor)
        score = output.sum()
        score.backward()

        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().cpu().detach().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, output.squeeze().cpu().detach().numpy()

def plot_grad_cam(input_tensor, cam_map, pred_anomaly, real_anomaly, save_path="grad_cam.png"):
    ndvi_input = input_tensor[0, 0].cpu().numpy()
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(ndvi_input, cmap='gray')
    axes[0].set_title('NDVI Input (Features)')

    heatmap = cv2.applyColorMap(np.uint8(255 * cam_map), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    axes[1].imshow(ndvi_input, cmap='gray', alpha=0.5)
    axes[1].imshow(heatmap, alpha=0.6)
    axes[1].set_title('Atenção da Rede (Grad-CAM)')

    im2 = axes[2].imshow(pred_anomaly, cmap='inferno')
    axes[2].set_title('Anomalia Prevista (Microclima)')
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    im3 = axes[3].imshow(real_anomaly, cmap='inferno')
    axes[3].set_title('Anomalia Real (Microclima)')
    fig.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def test_single_batch_overfit(model, train_loader, device, epochs=50):
    print("\n" + "="*70 + "\n🚀 TESTE DE SANIDADE: OVERFIT EM 1 BATCH\n" + "="*70)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    loss_fn = HybridPhysicsLoss(hotspot_w=0.1)

    seq, target, valid, std_anomaly = next(iter(train_loader))

    seq = seq.to(device)
    target = target.to(device)
    valid = valid.to(device)
    std_anomaly = std_anomaly.to(device)

    model_input = seq
    ndvi = seq[:, 0:1, :, :]

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        pred = model(model_input)
        loss = loss_fn(pred, target, ndvi, valid)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            with torch.no_grad():
                mae_c, hs_mae_c, iou = compute_metrics(pred, target, valid, std_anomaly)
            print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | MAE (°C): {mae_c:.4f} | HS MAE (°C): {hs_mae_c:.4f} | HS IoU: {iou:.4f}")

    print("="*70)

# ============================================================
# LOOP PRINCIPAL
# ============================================================
if __name__ == "__main__":
    all_cities = [d.name for d in Path(DATA_DIR).iterdir() if d.is_dir() and not d.name.startswith('.')]
    if len(all_cities) < K_FOLDS: exit("Cidades insuficientes para K-Fold.")

    all_files_raw = list(Path(DATA_DIR).rglob("*.npy"))
    valid_files_set = filter_cloudy_files_runtime(all_files_raw, temp_threshold=15.0, min_cloud_pct=30.0)

    if not valid_files_set: exit("Nenhum arquivo válido após o filtro de nuvens!")

    print("1. Extraindo Estatísticas Globais (Apenas Arquivos Válidos)...")
    base_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=all_cities, valid_files_set=valid_files_set, normalize=True, is_train=False)
    global_stats = (base_ds.mean, base_ds.std)

    kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

    temp_train = UrbanHeatDataset(DATA_DIR, allowed_cities=all_cities, valid_files_set=valid_files_set, stats=global_stats, is_train=True)
    temp_loader = DataLoader(temp_train, batch_size=BATCH_SIZE, shuffle=True)
    temp_model = HybridUrbanHeatNet(in_channels=4).to(DEVICE)
    test_single_batch_overfit(temp_model, temp_loader, DEVICE, epochs=50)
    del temp_model, temp_loader, temp_train

    print("\n2. Iniciando K-Fold Cross Validation Espacial...")
    for fold, (train_idx, val_idx) in enumerate(kf.split(all_cities)):
        train_cities = [all_cities[i] for i in train_idx]
        val_cities = [all_cities[i] for i in val_idx]

        train_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=train_cities, valid_files_set=valid_files_set, stats=global_stats, is_train=True)
        val_ds = UrbanHeatDataset(DATA_DIR, allowed_cities=val_cities, valid_files_set=valid_files_set, stats=global_stats, is_train=False)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

        model = HybridUrbanHeatNet(in_channels=4).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
        loss_fn = HybridPhysicsLoss(hotspot_w=0.1)
        best_loss = float('inf')

        print(f"\n--- Iniciando FOLD {fold + 1} ---")
        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0

            for seq, target, valid, std_anomaly in train_loader:
                seq, target, valid, std_anomaly = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE), std_anomaly.to(DEVICE)
                optimizer.zero_grad()

                pred = model(seq)

                loss = loss_fn(pred, target, seq[:, 0:1, :, :], valid)

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                train_loss += loss.item()

            model.eval()
            val_loss, val_mae_c, val_hs_mae_c, val_iou = 0, 0, 0, 0

            with torch.no_grad():
                for seq, target, valid, std_anomaly in val_loader:
                    seq, target, valid, std_anomaly = seq.to(DEVICE), target.to(DEVICE), valid.to(DEVICE), std_anomaly.to(DEVICE)

                    pred = model(seq)
                    val_loss += loss_fn(pred, target, seq[:, 0:1, :, :], valid).item()

                    mae_c, hs_mae_c, iou = compute_metrics(pred, target, valid, std_anomaly)
                    val_mae_c += mae_c
                    val_hs_mae_c += hs_mae_c
                    val_iou += iou

            n_t, n_v = max(len(train_loader), 1), max(len(val_loader), 1)
            train_loss /= n_t
            val_loss /= n_v

            scheduler.step(val_loss)

            print(f"Epoch {epoch+1:02d} | T.Loss: {train_loss:.4f} | V.Loss: {val_loss:.4f} | MAE (°C): {val_mae_c/n_v:.4f} | HS MAE (°C): {val_hs_mae_c/n_v:.4f} | IoU: {val_iou/n_v:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                torch.save(model.state_dict(), f"best_hybrid_fold_{fold}.pt")

        # GERAR GRAD-CAM DO MELHOR MODELO NO FOLD
        model.load_state_dict(torch.load(f"best_hybrid_fold_{fold}.pt", weights_only=True))
        cam_generator = RegressionGradCAM(model, target_layer=model.up1.conv.conv2)

        # Correção 1: Desempacotar 4 variáveis retornadas pelo dataloader
        sample_seq, sample_tgt, sample_valid, sample_std = next(iter(val_loader))

        # Correção 2: Slice de 4 canais ao invés de 5 para alinhar com o input da rede
        cam_map, pred_np = cam_generator.generate(sample_seq[0:1, :4, :, :].to(DEVICE))

        plot_grad_cam(sample_seq[0:1], cam_map, pred_np, sample_tgt[0, 0].numpy(), save_path=f"gradcam_fold_{fold}.png")
        print(f"-> Imagem do Grad-CAM salva como gradcam_fold_{fold}.png")

        break # Remova o break para rodar todos os folds

In [ ]:
import torch

# Substitua 'seu_modelo' pela classe que define sua rede
model = HybridUrbanHeatNet(in_channels=4)
checkpoint = torch.load("best_hybrid_fold_1.pt", map_location="cpu")

# Se o checkpoint for um dicionário contendo o state_dict:
model.load_state_dict(checkpoint)

# Para ver a arquitetura completa com as camadas e formatos:
print(model)